In [1]:
:set -XOverloadedStrings
:set -XScopedTypeVariables
:set -XRankNTypes
:set -XFlexibleInstances
:set -XFlexibleContexts
:set -XDeriveFunctor
:set -XGeneralizedNewtypeDeriving
import Data.List (sortBy, nub, minimumBy, maximumBy)
import Data.Ord (comparing, Down(..))
import Data.Maybe (fromMaybe, mapMaybe)
import Data.Map.Strict as Map(Map)
import qualified Data.Map.Strict as Map
import System.IO (hSetBuffering, stdout, BufferMode(..))
hSetBuffering stdout LineBuffering
putStrLn "\x2705 SETUP OK: Subjective Modeling (Pyt'ev)"

✅ SETUP OK: Subjective Modeling (Pyt'ev)

# ❓ Теория Субъективного Моделирования Пытьева

> **Источник:** Ю.П. Пытьев, «Математические методы субъективного моделирования в научных исследованиях»  
> **Часть 1** — Математические и эмпирические основы, Вестник МГУ, Физика. Астрономия, 2018, №1  
> **Часть 2** — Приложения, Вестник МГУ, Физика. Астрономия, 2018, №2

---

## Содержание

| № | Раздел | Уровень |
|---|--------|---------|
| 0 | Введение: связи с Kan и Toposes | Категорное |
| 1 | Пространство субъективной модели и неопределённый элемент | Пытьев |
| 2 | Дуальные шкалы $L$ и $\bar{L}$ | Пытьев |
| 3 | Меры правдоподобия $\mathrm{Pl}$ и доверия $\mathrm{Bel}$ | Пытьев |
| 4 | Принцип относительности — группа автоморфизмов $\Gamma$ | Пытьев |
| 5 | Pl-интегралы и Bel-интегралы | Пытьев |
| 6 | Субъективная независимость НОЭ | Пытьев |
| 7 | Условные субъективные распределения | Пытьев |
| 8 | Альтернативные варианты мер Pl и Bel | Пытьев |
| 9 | Эмпирические основы: восстановление модели | Пытьев |
| 10 | Комбинирование субъективных и эмпирических данных | Пытьев |
| 11 | Энтропия субъективной неопределённости (Часть 2) | Пытьев |
| 12 | Идентификация состояний НО.НЧ.О. (Часть 2) | Пытьев |
| 13 | $[0,1]$ как Quantale: внутренняя топологическая структура | Категорное |
| 14 | Двойственность Исбелла и гипотеза эквивалентности подходов | Категорное |


## 0. Введение и зависимости

**SubjectiveModeling** — синтез двух глубоких идей из предшествующих ноутбуков:
- **[Расширения Кана](KanExtensions.ipynb)**: меры Pl и Bel суть левое (Lan) и правое (Ran) расширения Кана;
- **[Топосы](Toposes.ipynb)**: $[0,1]$ является полной алгеброй Хейтинга (фреймом, квантальным пространством),
  а пара $(\Omega_+, \Omega_-) = (\mathrm{Pl}, \mathrm{Bel})$ образует **битопос**;
- **[Неопределённость](Uncertainty.ipynb)**: тот же формальный аппарат — это **теория возможностей Пытьева**
  ($\Pi, N$ с нормировкой $\sup\tau=1$); субъективное моделирование отличается лишь интерпретацией (раздел 9).

### Таблица ключевых связей

| Понятие в этом ноутбуке | Откуда | Раздел |
|------------------------|--------|--------|
| $\mathrm{Pl}(E) = \mathrm{Lan}_J\,\tau(E)$ | [Расширения Кана](KanExtensions.ipynb) | Lan — левое расширение |
| $\mathrm{Bel}(E) = \mathrm{Ran}_J\,\bar{\tau}(E)$ | [Расширения Кана](KanExtensions.ipynb) | Ran — правое расширение |
| $[0,1]$ как фрейм (полная алгебра Хейтинга) | [Топосы](Toposes.ipynb) | T4 — внутренняя логика |
| Битопос $(\Omega_+, \Omega_-) = (\mathrm{Pl}, \mathrm{Bel})$ | [Топосы](Toposes.ipynb) | T5, T9 — битопос |
| $\tau: X \to [0,1]$ как пресноп | [Топосы](Toposes.ipynb) | T1 — предпучки |
| Двойственность Исбелла | [Расширения Кана](KanExtensions.ipynb) | Ran/Lan над $[0,1]$-Frm |
| Теория возможностей $\Pi$, $N$ ($\sup\tau=1$) | [Неопределённость](Uncertainty.ipynb) | Раздел 9 — возможность/необходимость |
| Группа автоморфизмов $\Gamma$ | — | Принцип относительности Пытьева |

### Граф зависимостей

![Граф зависимостей](../diagrams/subj/subj_deps.svg)


In [2]:
-- Концептуальная карта: SubjectiveModeling зависит от...

-- Из KanExtensions.ipynb:
-- Lan_J tau (E) = sup_{x in E} tau(x) = Pl(E)
-- Ran_J tauBar (E) = inf_{x not in E} tauBar(x) = Bel(E)

-- Из Toposes.ipynb:
-- [0,1] -- complete Heyting algebra (quantale, фрейм)
-- BiTopos: два классификатора Omega+ и Omega- <=> Pl и Bel

-- Принцип относительности Пытьева:
-- Группа Gamma = автоморфизмы [0,1] как решётки
-- <=> инвариантность под Gamma = только порядок имеет смысл

putStrLn "Карта зависимостей загружена."


Карта зависимостей загружена.

### Связь с теорией возможностей

Аппарат субъективного моделирования (правдоподобие $\mathrm{Pl}$, доверие $\mathrm{Bel}$) и
аппарат **теории возможностей** Пытьева (возможность $\Pi$, необходимость $N$) — это
**одна и та же** математическая конструкция. Общие дуальные шкалы
$L=([0,1],\max,\min)$, $\bar L=([0,1],\min,\max)$; совпадают формулы
$$\mathrm{Pl}(A)=\Pi(A)=\sup_{x\in A}\tau(x), \qquad
\mathrm{Bel}(A)=N(A)=\inf_{x\notin A}\bar\tau(x),$$
и интегралы Сугено. Возможность и необходимость вводятся **совместно** на дуальных шкалах с
самого начала; обе функции $\tau,\bar\tau$ задаются формально одинаково в обеих теориях.

Различается только **интерпретация**:

| | Теория возможностей | Субъективное моделирование |
|--|--|--|
| Природа | объективная, частотная альтернатива вероятности | выражение интуиции эксперта |
| Источник $\tau,\bar\tau$ | эмпирические частоты | суждение эксперта (эмпирика — один из вариантов) |
| Приоритет | согласие с наблюдаемыми частотами | мнение эксперта |

Базовая теория возможностей (частотное построение, построение по вероятностям, пример
$w_n=1/p^n$) — в ноутбуке [Uncertainty.ipynb](Uncertainty.ipynb), раздел 9.

# 1. Пространство Субъективной Модели и Неопределённый Элемент

> **Зачем.** Прежде чем считать, нужно договориться, *что* мы моделируем: не частоту события, а **суждение исследователя** о неслучайном объекте. Этот раздел вводит носитель таких суждений — пару мер Pl/Bel на всех подмножествах $X$.

## Определение (Пытьев 2018, Часть 1, п. 1.1)

Модельер-исследователь (МИ) задаёт **субъективную модель** случайной величины $\tilde{x}$ с множеством значений $X$ как **пространство с мерами правдоподобия и доверия**:

$$\boxed{(X,\;\mathcal{P}(X),\;\mathrm{Pl}^{\tilde{x}},\;\mathrm{Bel}^{\tilde{x}})}$$

где $\mathcal{P}(X)$ — класс всех подмножеств $X$, а меры задаются распределениями:

$$\tau^{\tilde{x}}(x) = \mathrm{Pl}^{\tilde{x}}(\tilde{x} = x), \quad x \in X, \quad \sup_{x \in X} \tau^{\tilde{x}}(x) = 1$$

$$\bar{\tau}^{\tilde{x}}(x) = \mathrm{Bel}^{\tilde{x}}(\tilde{x} = x), \quad x \in X, \quad \inf_{x \in X} \bar{\tau}^{\tilde{x}}(x) = 0$$

и для любого $E \subseteq X$:

$$\mathrm{Pl}^{\tilde{x}}(E) \;=\; \sup_{x \in E}\, \tau^{\tilde{x}}(x), \qquad \mathrm{Pl}^{\tilde{x}}(\varnothing) = 0,\quad \mathrm{Pl}^{\tilde{x}}(X) = 1$$

$$\mathrm{Bel}^{\tilde{x}}(E) \;=\; \inf_{x \in X \setminus E}\, \bar{\tau}^{\tilde{x}}(x), \qquad \mathrm{Bel}^{\tilde{x}}(X) = 1,\quad \mathrm{Bel}^{\tilde{x}}(\varnothing) = 0$$

## Интерпретация

**Неопределённый элемент (НОЭ)** $\tilde{x}$ — это модель субъективных суждений МИ о неизвестном значении $x \in X$:

- $\mathrm{Pl}^{\tilde{x}}(\tilde{x} = x)$ — насколько **относительно правдоподобно** равенство $\tilde{x} = x$
- $\mathrm{Bel}^{\tilde{x}}(\tilde{x} \neq x)$ — насколько следует **относительно доверять** неравенству $\tilde{x} \neq x$

«Относительно» означает: численные значения мер в $(0,1)$ **не допускают абсолютного содержательного толкования** — существенна лишь их упорядоченность.

## НОЭ как неопределённая высказывательная переменная (п. 1.2)

В теоретико-множественном представлении логики высказываний:
- $X$ — множество элементарных высказываний
- $\mathcal{P}(X)$ — класс всех высказываний
- Любое высказывание $a$ взаимно однозначно представлено множеством $A \subseteq X$ тех элементарных высказываний $x$, каждое из которых влечёт $a$

Правдоподобие $\mathrm{Pl}^{\tilde{x}}(A)$ — **относительное правдоподобие истинности** неопределённого высказывания $\tilde{x} \in A$.  
Доверие $\mathrm{Bel}^{\tilde{x}}(A)$ — **относительное доверие** неравенству $\tilde{x} \notin X \setminus A$ (т.е. $\tilde{x} \in A$).

## Эквивалентность нечётким мерам (Замечание 1.1)

Пространство $(X, \mathcal{P}(X), \mathrm{Pl}^{\tilde{x}}, \mathrm{Bel}^{\tilde{x}})$ **формально эквивалентно** нечёткому пространству $(X, \mathcal{P}(X), \mathrm{P}, \mathrm{N})$ с мерами **возможности** $\mathrm{P}$ и **необходимости** $\mathrm{N}$.

**Границы.** Pl/Bel — не вероятность: нет аддитивности, и $\mathrm{Pl}(E)+\mathrm{Pl}(X\setminus E)$ не обязана равняться 1. Аппарат уместен, когда частотная база отсутствует или нестабильна; при стабильном вероятностном пространстве честнее работает теория вероятностей.


# 2. Дуальные Шкалы $L$ и $\bar{L}$ — Структуры Решётки

> **Зачем.** Значения Pl и Bel живут не в «числах», а в **шкалах с порядком**; их две, и они дуальны. Без этого раздела непонятно, почему min/max, а не +/×.

## Теорема о единственности операций (Пытьев 2018, п. 1.3)

Из **принципа относительности** следует, что бинарные операции в шкалах значений мер Pl и Bel **однозначно определяются** условиями непрерывности, коммутативности и граничными условиями.

### Шкала $L$ (для правдоподобия)

$$\boxed{L = ([0,1],\, +,\, \times) = ([0,1],\, \max,\, \min)}$$

$$a + b = \max\{a, b\}, \qquad a \times b = \min\{a, b\}, \qquad a, b \in [0,1]$$

Граничные условия: $a + 0 = a \times 1 = a$, $\;a + 1 = 1$, $\;a \times 0 = 0$.

### Дуальная шкала $\bar{L}$ (для доверия)

$$\boxed{\bar{L} = ([0,1],\, \bar{+},\, \bar{\times}) = ([0,1],\, \min,\, \max)}$$

$$a\,\bar{+}\,b = \min\{a,b\}, \qquad a\,\bar{\times}\,b = \max\{a,b\}, \qquad a, b \in [0,1]$$

Граничные условия: $a\,\bar{+}\,1 = 1$, $\;a\,\bar{\times}\,0 = 0$ и пр.

### Откуда берётся дуальность?

Шкалы $L$ и $\bar{L}$ связаны **дуальным изоморфизмом** $\theta \in \Theta$, где $\Theta$ — класс строго монотонных убывающих функций $\theta: [0,1] \to [0,1]$, $\theta(0) = 1$, $\theta(1) = 0$:

$$\mathrm{Bel}^{\tilde{x}}(A) = \theta\!\left(\mathrm{Pl}^{\tilde{x}}(X \setminus A)\right), \qquad \bar{\tau}^{\tilde{x}}(x) = \theta(\tau^{\tilde{x}}(x))$$

Такое соответствие называется **дуальной согласованностью** мер Pl и Bel.

## Структура решётки

Как $L$, так и $\bar{L}$ являются **полными решётками** (complete lattices):
- В $L$: $\sup = \max$, $\inf = \min$
- В $\bar{L}$: $\sup = \min$, $\inf = \max$

Пара $(L, \bar{L})$ образует **билатис** (bilattice): четыре операции $\max, \min, \bar{\max}, \bar{\min}$ на $[0,1]$ с двумя различными порядками.

**Границы.** Структуры решётки фиксируют только **порядок** — арифметика значений ($1-t$ и т.п.) есть лишь координатное представление (см. раздел 4).


# 3. Меры Правдоподобия $\mathrm{Pl}$ и Доверия $\mathrm{Bel}$

> **Зачем.** Главные операции теории: как из поточечных распределений $\tau,\bar\tau$ получаются меры событий. Здесь же — формальная стыковка с теорией возможностей: тот же аппарат $\Pi/N$ из [Uncertainty.ipynb](Uncertainty.ipynb) (раздел 9), но числа выражают интуицию эксперта, а не частоту.

## Полная аддитивность мер (Пытьев 2018, п. 1.1)

Меры $\mathrm{Pl}_{\tau}$ и $\mathrm{Bel}_{\bar{\tau}}$ **вполне аддитивны** в смысле операций шкал $L$ и $\bar{L}$:

$$\mathrm{Pl}_{\tau}\!\left(\bigcup_{j \in J} E_j\right) = \sup_{j \in J} \mathrm{Pl}_{\tau}(E_j) = \bigoplus_{j \in J}^{L} \mathrm{Pl}_{\tau}(E_j)$$

$$\mathrm{Bel}_{\bar{\tau}}\!\left(\bigcap_{j \in J} E_j\right) = \inf_{j \in J} \mathrm{Bel}_{\bar{\tau}}(E_j) = \bigoplus_{j \in J}^{\bar{L}} \mathrm{Bel}_{\bar{\tau}}(E_j)$$

## Правдоподобия характеристик ОИ (п. 1.4)

Любая функция $\varphi: X \to Y$ задаёт НОЭ $\tilde{y} = \varphi(\tilde{x})$ со значениями в $Y$ и пространство $(Y, \mathcal{P}(Y), \mathrm{Pl}^{\tilde{y}}, \mathrm{Bel}^{\tilde{y}})$, где:

$$\tau^{\tilde{y}}(y) = \mathrm{Pl}^{\tilde{y}}(\tilde{y} = y) = \sup_{x: \varphi(x)=y} \tau^{\tilde{x}}(x)$$

$$\bar{\tau}^{\tilde{y}}(y) = \mathrm{Bel}^{\tilde{y}}(\tilde{y} = y) = \inf_{x: \varphi(x) \neq y} \bar{\tau}^{\tilde{x}}(x)$$

Для любого $A \subseteq Y$:

$$\mathrm{Pl}^{\tilde{y}}(A) = \sup_{y \in A} \tau^{\tilde{y}}(y), \qquad \mathrm{Bel}^{\tilde{y}}(A) = \inf_{y \notin A} \bar{\tau}^{\tilde{y}}(y)$$

## Субъективные модели «абсолютного незнания» и «точного знания» (п. 1.5)

Инвариантные относительно выбора шкал $\gamma L$, $\bar{\gamma} \bar{L}$ модели:

$$\textbf{Абсолютное незнание:}\quad \tau^{\tilde{x}}(x) = 1,\; \bar{\tau}^{\tilde{x}}(x) = 0 \quad \forall x \in X$$

$$\Rightarrow \mathrm{Pl}^{\tilde{y}}(A) = 1,\; \mathrm{Bel}^{\tilde{y}}(A) = 0 \quad \forall \varphi: X \to Y,\; A \neq \varnothing$$

$$\textbf{Точное знание } x_0:\quad \tau^{\tilde{x}}(x) = \begin{cases} 1 & x = x_0 \\ 0 & x \neq x_0 \end{cases}, \quad \bar{\tau}^{\tilde{x}}(x) = \begin{cases} 1 & x = x_0 \\ 0 & x \neq x_0 \end{cases}$$

$$\Rightarrow \text{«точное знание» влечёт «точное знание» любого следствия модели}$$

**Границы.** $\sup$/$\inf$ делают меры **макситивными/минитивными**: вклад «многих средних свидетельств» не накапливается, как у суммы — это осознанная плата за работу без частот.


# 4. Принцип Относительности — Группа Автоморфизмов $\Gamma$

> **Зачем.** Если эксперт скажет «0.7», а другой «0.8» — значимо ли различие? Принцип относительности отвечает: инвариантен только **порядок**; конкретные числа — координаты.

## Группа $\Gamma$ (Пытьев 2018, п. 1.3)

**Группа автоморфизмов шкалы** $\Gamma$ — группа непрерывных строго монотонных функций:

$$\Gamma = \{\gamma: [0,1] \to [0,1] \mid \gamma(0) = 0,\; \gamma(1) = 1,\; \gamma \text{ строго монотонно возрастает}\}$$

с групповой операцией $(\gamma \circ \gamma')(a) = \gamma(\gamma'(a))$.

## Принцип относительности

Меры $\mathrm{Pl}^{\tilde{x}}$ и $\mathrm{Pl}'^{\tilde{x}}$ **эквивалентны**, если:

$$\exists\, \gamma \in \Gamma\; \forall E \in \mathcal{P}(X):\quad \gamma(\mathrm{Pl}^{\tilde{x}}(E)) = \mathrm{Pl}'^{\tilde{x}}(E)$$

Каждый МИ может формулировать модель в **своей шкале** $\gamma L$, $\bar{\gamma} \bar{L}$, $\gamma, \bar{\gamma} \in \Gamma$.

## Инвариантность автоморфизмов

Для любого $\gamma \in \Gamma$ и любых $a, b \in [0,1]$:

$$\gamma(a + b) = \gamma(a) + \gamma(b), \qquad \gamma(a \times b) = \gamma(a) \times \gamma(b)$$

$$a < b \;\Leftrightarrow\; \gamma(a) < \gamma(b)$$

Это означает: $\gamma$ — **автоморфизм решётки** $(L, \max, \min)$.

## Следствия принципа относительности

1. Численные значения $\mathrm{Pl}(E) \in (0,1)$ **не имеют абсолютного смысла** — только упорядоченность инвариантна
2. Содержательны лишь равенства $\mathrm{Pl}(E) = 0$ или $\mathrm{Pl}(E) = 1$ (не зависят от $\gamma$)
3. МИ может выбрать $\gamma$ так, чтобы значения мер были сколь угодно близки к 0 или 1

## Группы $\Gamma_S$ с выделенными значениями (п. 1.9.1)

Если коллективу МИ нужно **содержательно интерпретировать** некоторые значения $\alpha_i \in (0,1)$, они договариваются использовать подгруппу:

$$\Gamma_S = \{\gamma \in \Gamma \mid \gamma(\alpha_i) = \alpha_i,\; i = 1,\ldots,n\}$$

В частности, $\Gamma_{\{1/2\}}$ оставляет неподвижным значение $1/2$ (индифферентность МИ).

## Третий вариант: психофизическая группа (п. 1.9.2)

Группа $\Gamma'$ порождена преобразованиями $\gamma_\alpha(a) = a^\alpha$, $\alpha > 0$ (степенные функции Стивенса):

$$\gamma_\alpha(a + ' b) = \gamma_\alpha(a) +' \gamma_\alpha(b), \qquad \gamma_\alpha(a \times' b) = \gamma_\alpha(a) \times' \gamma_\alpha(b)$$

$\Gamma'$-инвариант: $\dfrac{\log \mathrm{Pl}'(A)}{\log \mathrm{Pl}'(B)} = \text{const}$ — допускает психофизическую интерпретацию.

**Границы.** Инвариантность к $\Gamma$ означает, что выводы, зависящие от арифметики значений (а не порядка), методологически нелегитимны в этой теории.


# 5. Pl-Интегралы и Bel-Интегралы

> **Зачем.** Чтобы принимать решения, нужны «средние» — аналоги интеграла Лебега. Здесь строятся интегралы по неаддитивным мерам (родственники интеграла Сугено).

## Определение 1.1 (Пытьев 2018, п. 1.6)

Пусть $\mathcal{L}(X)$ — класс функций $g: X \to L$ и $\bar{\mathcal{L}}(X)$ — класс функций $\bar{g}: X \to \bar{L}$.

**pl-интеграл** $\mathrm{pl}(\cdot): \mathcal{L}(X) \to L$ — **однородный и вполне аддитивный** функционал:

- **Однородность:** $\mathrm{pl}((a \times g)(\cdot)) = a \times \mathrm{pl}(g(\cdot))$ для любого $a \in [0,1]$
- **Полная аддитивность:** $\mathrm{pl}\!\left(\sum_j g_j\right)(\cdot) = \sum_j \mathrm{pl}(g_j(\cdot))$ в $L$

Аналогично **bel-интеграл** $\mathrm{bel}(\cdot): \bar{\mathcal{L}}(X) \to \bar{L}$ однороден и вполне аддитивен.

## Теорема 1.1: явное представление (Пытьев 2018, п. 1.6)

Для любого pl-интеграла $\exists!\; \tau: X \to L$ такая, что:

$$\boxed{\mathrm{pl}_{\tau}(g(\cdot)) = \sup_{x \in X} \min\{\tau(x),\, g(x)\}}$$

Для любого bel-интеграла $\exists!\; \bar{\tau}: X \to \bar{L}$ такая, что:

$$\boxed{\mathrm{bel}_{\bar{\tau}}(\bar{g}(\cdot)) = \inf_{x \in X} \max\{\bar{\tau}(x),\, \bar{g}(x)\}}$$

## Связь с мерами Pl и Bel

$$\mathrm{Pl}(E) = \mathrm{pl}_{\tau}(\chi_E(\cdot)) = \sup_{x \in E} \tau(x)$$

$$\mathrm{Bel}(E) = \mathrm{bel}_{\bar{\tau}}(\chi_E(\cdot)) = \inf_{x \in X \setminus E} \bar{\tau}(x)$$

## Интерпретация интегралов

Для функции потерь или ценности $f: X \to [0,1]$:

$$\mathrm{pl}_{\tau}(f) = \sup_{x \in X} \min\{\tau(x), f(x)\} \qquad \text{(«оптимистический» — наилучший сценарий)}$$

$$\mathrm{bel}_{\bar{\tau}}(f) = \inf_{x \in X} \max\{\bar{\tau}(x), f(x)\} \qquad \text{(«пессимистический» — наихудший сценарий)}$$

**Границы.** Pl/Bel-интегралы идемпотентны и устойчивы к выбросам, но «не чувствуют» массу повторений — там, где важна аккумуляция, нужен Лебег по вероятности.


In [3]:
-- | Субъективная модель Пытьева: базовые типы и операции (Разделы 1-5)

-- Тип НОЭ (неопределённый элемент)
data SubjModel a = SubjModel
  { smDomain :: [a]
  , smTau    :: a -> Double   -- распределение правдоподобий tau(x) = Pl(x=x)
  , smTauBar :: a -> Double   -- распределение доверий tauBar(x) = Bel(x=x)
  }

-- | Мера правдоподобия: Pl(E) = sup_{x in E} tau(x)
plMeasure :: [a] -> (a -> Double) -> Double
plMeasure xs tau = maximum (0 : map tau xs)

-- | Мера доверия: Bel(E) = inf_{x not in E} tauBar(x)
belMeasure :: [a] -> (a -> Double) -> Double
belMeasure xsComp tauBar = minimum (1 : map tauBar xsComp)

-- | pl-интеграл: sup_{x in X} min(tau(x), g(x))
plIntegral :: [a] -> (a -> Double) -> (a -> Double) -> Double
plIntegral xs tau g = maximum (0 : [min (tau x) (g x) | x <- xs])

-- | bel-интеграл: inf_{x in X} max(tauBar(x), gBar(x))
belIntegral :: [a] -> (a -> Double) -> (a -> Double) -> Double
belIntegral xs tauBar gBar = minimum (1 : [max (tauBar x) (gBar x) | x <- xs])

-- | Абсолютное незнание: tau(x)=1, tauBar(x)=0 для всех x
absoluteIgnorance :: [a] -> SubjModel a
absoluteIgnorance xs = SubjModel xs (const 1.0) (const 0.0)

-- | Точное знание x0: tau(x0)=1, tau(x)=0 для x/=x0
exactKnowledge :: Eq a => [a] -> a -> SubjModel a
exactKnowledge xs x0 = SubjModel xs pl bel
  where
    pl x = if x == x0 then 1.0 else 0.0
    bel x = if x == x0 then 1.0 else 0.0

-- | Образ НОЭ под функцией phi: X -> Y
imageModel :: Eq b => SubjModel a -> (a -> b) -> [b] -> SubjModel b
imageModel sm phi ys = SubjModel ys newTau newTauBar
  where
    xs = smDomain sm
    newTau y = maximum (0 : [smTau sm x | x <- xs, phi x == y])
    newTauBar y = minimum (1 : [smTauBar sm x | x <- xs, phi x /= y])

-- | Применение группы автоморфизмов Gamma к мере Pl
applyGamma :: (Double -> Double) -> SubjModel a -> SubjModel a
applyGamma gamma sm = sm { smTau = gamma . smTau sm }

-- | Дуальная согласованность через theta = (1 - .)
isDuallyConsistent :: Eq a => SubjModel a -> Bool
isDuallyConsistent sm = all check (smDomain sm)
  where
    check x = abs (smTauBar sm x - (1.0 - smTau sm x)) < 1e-9

-- Демонстрация
data State = SA | SB | SC deriving (Show, Eq, Ord, Enum, Bounded)

demoS15 :: IO ()
demoS15 = do
  let xs = [SA, SB, SC]
      -- МИ задаёт распределение правдоподобий
      tau SA = 1.0; tau SB = 0.7; tau SC = 0.3
      -- Дуально согласованное через theta(t) = 1-t
      tauBar x = 1.0 - tau x
      sm = SubjModel xs tau tauBar
      subE = [SA, SB]
      compl = [SC]
  putStrLn "=== Раздел 1-5: Субъективная модель Пытьева ==="
  putStrLn $ "  tau(A)=1, tau(B)=0.7, tau(C)=0.3"
  putStrLn $ "  Pl({A,B})   = " ++ show (plMeasure subE tau)
  putStrLn $ "  Bel({A,B})  = " ++ show (belMeasure compl tauBar)
  putStrLn $ "  pl-инт f    = " ++ show (plIntegral xs tau (\v -> case v of SA->0.8; SB->0.5; SC->0.2))
  putStrLn $ "  bel-инт f   = " ++ show (belIntegral xs tauBar (\v -> case v of SA->0.2; SB->0.5; SC->0.8))
  putStrLn $ "  Дуально согл: " ++ show (isDuallyConsistent sm)
  putStrLn ""
  let igSm = absoluteIgnorance xs
  putStrLn "  Абсолютное незнание:"
  putStrLn $ "    Pl(X) = " ++ show (plMeasure xs (smTau igSm))
  putStrLn $ "    Pl({}) = " ++ show (plMeasure [] (smTau igSm))
  let exSm = exactKnowledge xs SA
  putStrLn "  Точное знание SA:"
  putStrLn $ "    Pl({SA}) = " ++ show (plMeasure [SA] (smTau exSm))
  putStrLn $ "    Pl({SB}) = " ++ show (plMeasure [SB] (smTau exSm))

demoS15

Line 69: Redundant $
Found:
putStrLn $ "  tau(A)=1, tau(B)=0.7, tau(C)=0.3"
Why not:
putStrLn "  tau(A)=1, tau(B)=0.7, tau(C)=0.3"Line 72: Use lambda-case
Found:
\ v
  -> case v of
       SA -> 0.8
       SB -> 0.5
       SC -> 0.2
Why not:
\case
  SA -> 0.8
  SB -> 0.5
  SC -> 0.2Line 73: Use lambda-case
Found:
\ v
  -> case v of
       SA -> 0.2
       SB -> 0.5
       SC -> 0.8
Why not:
\case
  SA -> 0.2
  SB -> 0.5
  SC -> 0.8

=== Раздел 1-5: Субъективная модель Пытьева ===
  tau(A)=1, tau(B)=0.7, tau(C)=0.3
  Pl({A,B})   = 1.0
  Bel({A,B})  = 0.7
  pl-инт f    = 0.8
  bel-инт f   = 0.2
  Дуально согл: True

  Абсолютное незнание:
    Pl(X) = 1.0
    Pl({}) = 0.0
  Точное знание SA:
    Pl({SA}) = 1.0
    Pl({SB}) = 0.0

# 6. Субъективная Независимость НОЭ

> **Зачем.** Совместные модели нескольких неопределённых элементов требуют понятия независимости — здесь оно задаётся через min/max вместо произведения.

## Определение 1.2 (Пытьев 2018, п. 1.7)

Пусть $\tilde{z} = (\tilde{z}_1, \ldots, \tilde{z}_n)$ — НОЭ со значениями в $Z_1 \times \cdots \times Z_n$.

**НОЭ $\tilde{z}_1, \ldots, \tilde{z}_n$ взаимно $\mathrm{Pl}$-независимы**, если правдоподобие события «$\forall i: \tilde{z}_i = z_i$»:

$$\tau^{\tilde{z}_1, \ldots, \tilde{z}_n}(z_1, \ldots, z_n) = \min_{1 \leq i \leq n} \tau^{\tilde{z}_i}(z_i) = \bigtimes_{i=1}^{n} \tau^{\tilde{z}_i}(z_i)$$

**НОЭ $\tilde{z}_1, \ldots, \tilde{z}_n$ взаимно $\mathrm{Bel}$-независимы**, если:

$$\bar{\tau}^{\tilde{z}_1, \ldots, \tilde{z}_n}(z_1, \ldots, z_n) = \max_{1 \leq i \leq n} \bar{\tau}^{\tilde{z}_i}(z_i) = \bar{\bigtimes}_{i=1}^{n} \bar{\tau}^{\tilde{z}_i}(z_i)$$

## Независимость событий

**События** $B_1, \ldots, B_n \in \mathcal{P}(Y)$ **взаимно $\mathrm{Pl}$-независимы**, если:

$$\mathrm{Pl}\!\left(\bigcap_{i=1}^n B_i\right) = \min_{1 \leq i \leq n} \mathrm{Pl}(B_i)$$

**$\mathrm{Bel}$-независимы**, если:

$$\mathrm{Bel}\!\left(\bigcup_{i=1}^n B_i\right) = \max_{1 \leq i \leq n} \mathrm{Bel}(B_i)$$

## Субъективная независимость (Определение 1.3)

**События $B_1, B_2$ субъективно $\mathrm{Pl}$-независимы**, если $\exists$ непрерывная $f: [0,1]^2 \to [0,1]$ такая, что:

$$\mathrm{Pl}(B_1 \cap B_2) = f(\mathrm{Pl}(B_1), \mathrm{Pl}(B_2))$$

**Теорема Пытьева:** взаимная независимость и субъективная независимость **эквивалентны**, причём $f(a,b) = \min(a,b)$.

## Связь Pl- и Bel-независимостей

Если меры Pl и Bel **дуально согласованы** ($\exists \theta \in \Theta: \mathrm{Bel}(A) = \theta(\mathrm{Pl}(X \setminus A))$), то $\mathrm{Pl}$- и $\mathrm{Bel}$-независимости **эквивалентны**.

**Границы.** Pl-независимость не влечёт Bel-независимость и наоборот; интуиция вероятностной независимости переносится лишь частично.


# 7. Условные Субъективные Распределения

> **Зачем.** Обновление суждений по наблюдению — субъективный аналог условной вероятности и основа применений (диагностика в разделе 16).

## Условное распределение правдоподобий (Пытьев 2018, п. 1.8)

Пусть $\tilde{z} = (\tilde{z}_1, \tilde{z}_2)$, $z_1 \in Z_1$, $z_2 \in Z_2$.

**Вариантом условного распределения правдоподобий** $z_1$ при условии $z_2 = z_2$ называется любое решение уравнения:

$$\min\{\tau^{\tilde{z}_1 | \tilde{z}_2}(z_1 | z_2),\; \tau^{\tilde{z}_2}(z_2)\} = \tau^{\tilde{z}_1, \tilde{z}_2}(z_1, z_2)$$

где $\tau^{\tilde{z}_2}(z_2) = \sup_{z_1 \in Z_1} \tau^{\tilde{z}_1, \tilde{z}_2}(z_1, z_2)$.

## Проблема субъективной шкалы (Определение 1.6)

Когда $0 < \tau^{\tilde{z}_2}(z_2) < 1$, условное распределение может быть не распределением правдоподобий в шкале $L$.

Решение: **субъективная шкала** $\gamma_{z_2} L$, где $\gamma_{z_2}: [0, \tau^{\tilde{z}_2}(z_2)] \to [0,1]$ — строго монотонная функция с $\gamma_{z_2}(0)=0$, $\gamma_{z_2}(\tau^{\tilde{z}_2}(z_2))=1$.

Тогда **условное правдоподобие** в субъективной шкале $\gamma_{z_2} L$:

$$\tau^{\tilde{z}_1 | \tilde{z}_2}_s(z_1 | z_2) = \gamma_{z_2} \circ \tau^{\tilde{z}_1, \tilde{z}_2}(z_1, z_2), \qquad z_1 \in Z_1$$

является распределением правдоподобий в $\gamma_{z_2} L$, в котором $z_2 = z_2$ — достоверное событие.

## Аналогия с условной вероятностью

В теории вероятностей: $\mathrm{Pr}(A | B) = \mathrm{Pr}(A \cap B) / \mathrm{Pr}(B)$ — нормирующий множитель $1/\mathrm{Pr}(B)$ задаёт «субъективную шкалу».

В теории Пытьева: переход в шкалу $\gamma_{z_2} L$ играет ту же роль, но в рамках принципа относительности.

**Границы.** Условные распределения определяются неоднозначно (есть варианты определения); выбор варианта — модельное решение.


# 8. Альтернативные Варианты Мер Pl и Bel

> **Зачем.** Первая пара (sup-min) — не единственная согласованная конструкция; здесь — систематика альтернатив и критерии выбора.

## Три варианта теории (Пытьев 2018, п. 1.9)

Пытьев рассматривает три варианта мер правдоподобия и доверия.

### Первый вариант (основной)

$$a + b = \max\{a,b\}, \quad a \times b = \min\{a,b\}$$

Группа $\Gamma$ — все непрерывные строго монотонные функции $[0,1] \to [0,1]$.  
**Числовые значения** мер, отличные от 0 и 1, **не допускают содержательной интерпретации**.

### Второй вариант (с фиксированными точками, п. 1.9.1)

Для содержательной интерпретации значений $\alpha_1, \ldots, \alpha_n \in (0,1)$ используется подгруппа $\Gamma_S \subset \Gamma$, оставляющая эти значения неподвижными.

Проекторы на интервалы $[\alpha_i, \alpha_{i+1}]$:

$$(u)^{\alpha} = \max\{\alpha, u\}, \qquad (u)_{\alpha} = \min\{\alpha, u\}$$

Шкала $L_{S'}$ разбивается на интервалы $[\alpha_i, \alpha_{i+1}]$, $i = 0, \ldots, n$, с группой $\Gamma_{S'} = \Gamma_{[\alpha_0,\alpha_1]} \otimes \cdots \otimes \Gamma_{[\alpha_n,\alpha_{n+1}]}$.

МИ могут содержательно интерпретировать **принадлежность** значений pl-интеграла неподвижным интервалам.

### Третий вариант (психофизический, п. 1.9.2)

Операции в шкале $L'$:

$$a +' b = \max\{a,b\}, \qquad a \times' b = a \cdot b \; (\text{обычное произведение})$$

Группа $\Gamma'$ порождена $\gamma_\alpha(a) = a^\alpha$, $\alpha > 0$.

Шкала $\bar{L}'$ значений $\mathrm{Bel}'$ — дуально изоморфная $L'$:

$$\theta_\beta(u) = \log_\beta u^{-1}, \quad 0 < u \leq 1, \qquad \bar{v} +' \bar{w} = \min\{\bar{v}, \bar{w}\}, \qquad \bar{v} \times' \bar{w} = \bar{v} + \bar{w}$$

Интегралы третьего варианта:

$$\mathrm{pl}'_{\tau}(f) = \max_{x \in X} (\tau(x) \cdot f(x))$$

$$\mathrm{bel}'_{\bar{\tau}}(\bar{g}) = \min_{x \in X} (\log_\beta \tau(x)^{-1} + \bar{g}(x))$$

**Инвариант третьего варианта** (допускает содержательную интерпретацию):

$$\frac{\log \mathrm{Pl}'(A)}{\log \mathrm{Pl}'(B)} = \text{const при всех } \gamma_\alpha \in \Gamma'$$

Это **отношение степеней** вероятностных оценок — аналог психофизических функций Стивенса.

**Границы.** Разные варианты дают разные численные ответы при одинаковых данных — сравнивать результаты можно только внутри одного варианта.


# 9. Эмпирические Основы: Восстановление Модели НО.НЧ.О.

> **Зачем.** Откуда брать $\tau$? Раздел показывает, как восстановить модель из упорядочивающих ответов эксперта/наблюдений — мост от слов к числам.

## Нечёткий неопределённый объект (НО.НЧ.О.)

**Неопределённый нечёткий объект** — объект, моделью которого является нечёткое пространство $(Y, \mathcal{P}(Y), \mathrm{P}(\cdot; x), \mathrm{N}(\cdot; x))$, зависящее от **неизвестного параметра** $x \in X$.

МИ предлагает субъективную модель НОЭ $\tilde{x}$ — пространство $(X, \mathcal{P}(X), \mathrm{Pl}^{\tilde{x}}, \mathrm{Bel}^{\tilde{x}})$.

## Эмпирическое оценивание по данным наблюдений (п. 2.1.2)

Если МИ доступны данные наблюдений $\eta = y_0 \in Y$ за НО.НЧ.О., то **нечёткий неопределённый элемент (НЧ.НОЭ)** $\hat{x} = \hat{x}(\eta)$, эмпирически оценивающий параметр $x \in X$, задаётся:

$$\tau^{\hat{x}}(x; y_0) = \gamma \circ g^{\eta}(y_0; x)$$

$$\bar{\tau}^{\hat{x}}(x; y_0) = \theta \circ g^{\eta}(y_0; x)$$

где $g^{\eta}(y; x)$ — распределение возможностей наблюдения $\eta = y$ при параметре $x$, $\gamma \in \Gamma$, $\theta \in \Theta$.

## Семейства оценивающих множеств максимального правдоподобия (п. 2.1.2)

$$\Phi^{-1}(y_0; \Lambda) = \{x \in X: g^{\eta}(y_0; x) \geq \min\{\Lambda, \max_{x' \in X} g^{\eta}(y_0; x')\}\}, \qquad \Lambda \in (0,1)$$

Вариант правдоподобия $\tau^{\hat{x}}(x; y_0) = \gamma \circ g^{\eta}(y_0; x)$ — **нечёткое правдоподобие** равенства $x = x \in X$ при наблюдении $\eta = y_0$.

## Правдоподобие согласия модели с данными (п. 2.3)

$$\mathrm{Pl}^{\tilde{x}}(\tilde{x} \sim \eta) = 1 - \sup\{\Lambda \in (0,1): \mathrm{Pl}^{\tilde{x}}(x \in \Phi^{-1}(\eta)) = 1\}$$

Чем меньше это значение, тем значительнее наблюдение $\eta$ **свидетельствует против** субъективной модели $(X, \mathcal{P}(X), \mathrm{Pl}^{\tilde{x}}, \mathrm{Bel}^{\tilde{x}})$.

**Границы.** Восстановление определено с точностью до $\Gamma$ (порядок!), поэтому претензии на «точные значения» здесь бессмысленны.


# 10. Комбинирование Субъективных и Эмпирических Данных

> **Зачем.** На практике есть и мнения экспертов, и данные; нужен способ их честно склеивать (используется в примере с двигателем, раздел 16).

## Задача согласованности (Пытьев 2018, п. 2.2)

Пусть $X = \{x_1, \ldots, x_m\}$, а $\tau^{(0)}(x_i)$ и $\tau^{(1)}(x_i)$ — субъективное и эмпирическое распределения правдоподобий.

Поскольку распределения задаются **в разных шкалах**, их нельзя непосредственно сравнивать числами. Можно сравнивать только **упорядоченности** значений.

## Матрица парных сравнений

Каждому распределению $\tau^{(a)}(x_i)$, $i = 1, \ldots, m$, $a = 0, 1$, сопоставляется **матрица парных сравнений** $M(a) = \{m^{(a)}_{kl}\}$:

$$m^{(a)}_{kl} = \begin{cases} 1 & \text{если } \tau^{(a)}_k > \tau^{(a)}_l \\ 0 & \text{если } \tau^{(a)}_k = \tau^{(a)}_l \\ -1 & \text{если } \tau^{(a)}_k < \tau^{(a)}_l \end{cases}$$

## Евклидово расстояние между матрицами

На классе $\mathcal{M}_{m+1}$ всех матриц парных сравнений вводится расстояние:

$$\rho(M', M'') = \left(\sum_{k,l=1}^{m+1} (m'_{kl} - m''_{kl})^2\right)^{1/2}$$

## Оптимальное совместное распределение (задача минимизации)

$$M^* = \arg\min_{M \in \mathcal{M}_{m+1}} \sum_{a=0,1} w_a^2 \rho^2(M(a), M)$$

где $w_a$ — «веса» субъективного и эмпирического распределений.

**Задача эквивалентна** нахождению матрицы парных сравнений, ближайшей к средневзвешенной матрице:

$$\bar{M} = w_0 M(0) + w_1 M(1), \qquad w_0 + w_1 = 1$$

Элементы $M^*$: $m^*_{kl} = \mathrm{sign}(\bar{m}_{kl})$ при $|\bar{m}_{kl}| \geq 1/2$, иначе $0$.

## Критерий согласованности

Если $M^* \notin \mathcal{M}_{m+1}$ (не является матрицей парных сравнений) — уровень противоречия субъективных и эмпирических данных **превышает критический**; матрице $M^*$ доверять не следует.

**Границы.** Комбинирование чувствительно к согласованности источников; противоречивые эксперты требуют отдельной обработки (взвешивание, отбраковка).


In [4]:
-- | Субъективная независимость, условные распределения,
-- | эмпирическое комбинирование (Разделы 6-10)

-- Повторяем базовый тип
data SubjModel a = SubjModel
  { smDomain :: [a]
  , smTau    :: a -> Double
  , smTauBar :: a -> Double
  }

-- | Pl-независимость НОЭ x1, x2: tau_{x1,x2}(a,b) = min(tau1(a), tau2(b))
plJointDist :: SubjModel a -> SubjModel b -> (a,b) -> Double
plJointDist sm1 sm2 (a,b) = min (smTau sm1 a) (smTau sm2 b)

-- | Bel-независимость: tauBar_{x1,x2}(a,b) = max(tBar1(a), tBar2(b))
belJointDist :: SubjModel a -> SubjModel b -> (a,b) -> Double
belJointDist sm1 sm2 (a,b) = max (smTauBar sm1 a) (smTauBar sm2 b)

-- | Матрица парных сравнений распределения
compMatrix :: [Double] -> [[Int]]
compMatrix vals = [[cmp vi vj | vj <- vals] | vi <- vals]
  where
    cmp a b
      | a > b    =  1
      | a < b    = -1
      | otherwise =  0

-- | Расстояние между матрицами парных сравнений
matrixDist :: [[Int]] -> [[Int]] -> Double
matrixDist m1 m2 = sqrt . fromIntegral $ sum
  [ (m1 !! i !! j - m2 !! i !! j)^2
  | i <- [0..length m1-1], j <- [0..length (m1!!0)-1] ]

-- | Оптимальное совместное распределение (задача 2.2 Пытьева)
-- Возвращает ранги (позиции по убыванию)
combineDistributions :: [Double] -> [Double] -> Double -> Double -> [Double]
combineDistributions subj empi w0 w1 =
  let mSubj = compMatrix subj
      mEmpi = compMatrix empi
      n = length subj
      -- Взвешенная средняя матрица (значения как Double)
      mBar i j = w0 * fromIntegral (mSubj!!i!!j) + w1 * fromIntegral (mEmpi!!i!!j)
      -- Ранги: позиция каждого элемента (число строк где элемент i лучше)
      rank i = fromIntegral $ length [j | j <- [0..n-1], mBar i j > 0]
  in [rank i | i <- [0..n-1]]

-- Демонстрация
demoS610 :: IO ()
demoS610 = do
  putStrLn "=== Раздел 6-10: Независимость и Комбинирование ==="
  -- Pl-независимость
  let tau1 x = case (x::Int) of 1->1.0; 2->0.5; _->0.3
      tau2 y = if (y::Bool) then 1.0 else 0.4
      sm1 = SubjModel [1,2,3::Int] tau1 (\x -> 1 - tau1 x)
      sm2 = SubjModel [True,False] tau2 (\y -> 1 - tau2 y)
  putStrLn $ "  tau_{x1,x2}(1,True)  = min(1.0, 1.0) = " ++ show (plJointDist sm1 sm2 (1,True))
  putStrLn $ "  tau_{x1,x2}(2,False) = min(0.5, 0.4) = " ++ show (plJointDist sm1 sm2 (2,False))
  -- Комбинирование
  let subj = [1.0, 0.8, 0.5, 0.2]
      empi = [0.9, 0.7, 0.6, 0.3]
      combined = combineDistributions subj empi 0.5 0.5
  putStrLn ""
  putStrLn "  Комбинирование субъективного и эмпирического:"
  putStrLn $ "    Субъективное: " ++ show subj
  putStrLn $ "    Эмпирическое: " ++ show empi
  putStrLn $ "    Ранги после комбинирования: " ++ show combined
  putStrLn $ "    Расстояние M(subj)--M(empi): " ++
             show (matrixDist (compMatrix subj) (compMatrix empi))

demoS610

Line 32: Use head
Found:
m1 !! 0
Why not:
head m1

=== Раздел 6-10: Независимость и Комбинирование ===
  tau_{x1,x2}(1,True)  = min(1.0, 1.0) = 1.0
  tau_{x1,x2}(2,False) = min(0.5, 0.4) = 0.4

  Комбинирование субъективного и эмпирического:
    Субъективное: [1.0,0.8,0.5,0.2]
    Эмпирическое: [0.9,0.7,0.6,0.3]
    Ранги после комбинирования: [3.0,2.0,1.0,0.0]
    Расстояние M(subj)--M(empi): 0.0

# 11. Энтропия Субъективной Неопределённости (Часть 2)

> **Зачем.** Сколько неопределённости в модели? Аналог шенноновской энтропии в min/max-арифметике даёт меру информативности суждений.

## Шенноновская энтропия как отправная точка (Пытьев 2018, Часть 2, п. 2.1)

Шеннон определил энтропию как **среднюю информативность** случайного исхода испытания $(X, \mathcal{P}(X), \mathrm{Pr})$:

$$H(\mathrm{pr}_{\cdot}) = \sum_{i=1}^k \mathrm{pr}_i \log_a \frac{1}{\mathrm{pr}_i}$$

Связь с законом больших чисел (ЗБЧ): число «типичных последовательностей» длины $n$ растёт как $a^{nH}$.

## Информативность и неопределённость НОЭ (п. 2.2)

Для НОЭ $\tilde{x}$ с распределениями $\tau^{\tilde{x}}$ и $\bar{\tau}^{\tilde{x}}$ Пытьев определяет пару энтропий — формальных аналогов шенноновской:

**Информативность** НОЭ $\tilde{x}$ (в первом варианте мер Pl, Bel):

$$\boxed{H(\tau^{\tilde{x}}) = \sup_{x \in X} \min\{\tau^{\tilde{x}}(x),\; \bar{\tau}^{\tilde{x}}(x)\} = \mathrm{pl}_{\tau}(\bar{\tau}(\cdot))}$$

**Неопределённость** НОЭ $\tilde{x}$:

$$\boxed{H(\bar{\tau}^{\tilde{x}}) = \inf_{x \in X} \max\{\bar{\tau}^{\tilde{x}}(x),\; \tau^{\tilde{x}}(x)\} = \mathrm{bel}_{\bar{\tau}}(\tau(\cdot))}$$

Если меры Pl и Bel **дуально согласованы** ($\bar{\tau} = \theta \circ \tau$), то:

$$H_\theta(\tau) = \mathrm{pl}_{\tau}(\theta \circ \tau(\cdot)) = \sup_{x \in X} \min\{\tau(x),\; \theta(\tau(x))\}$$

## Свойства энтропий (формальное подобие шенноновским)

Для независимых НОЭ $\tilde{x}_1$ и $\tilde{x}_2$ (с Pl-независимым совместным распределением):

$$H_\theta(\tau^{\tilde{x}_1} \times \tau^{\tilde{x}_2}) = \max\{H_\theta(\tau^{\tilde{x}_1}),\; H_\theta(\tau^{\tilde{x}_2})\}$$

Независимо повторённое суждение **не несёт дополнительной информации** (в отличие от шенноновской, которая удваивается!). Причина: **отсутствие ЗБЧ** в первом варианте мер.

## Энтропия в третьем варианте (п. 2.3) — со своим ЗБЧ

В третьем варианте мер $\mathrm{Pl}'$ (с $\gamma_\alpha(a) = a^\alpha$):

$$H'(\tau^{\tilde{x}}) = \max_{x \in X} \bigl(\tau(x) \cdot \log_\beta \tau(x)^{-1}\bigr)$$

Здесь есть аналог ЗБЧ, и математическое ожидание субъективной информативности **связано с шенноновской энтропией**:

$$\sum_{i=1}^k \frac{\mathrm{pr}_i}{\mathrm{pr}_1} \cdot \log_a \frac{\mathrm{pr}_i}{\mathrm{pr}_1} = \delta \cdot H(\mathrm{pr}_{\cdot})$$

где $\delta = \lim_{n \to \infty} \delta(n)$ — параметр связи субъективных и вероятностных шкал.

**Границы.** Это формальный аналог: без ЗБЧ интерпретации «числа типичных последовательностей» нет; величина осмыслена как порядковая характеристика.


# 12. Идентификация Состояний НО.НЧ.О. — Оптимальное Правило (Часть 2)

> **Зачем.** Кульминация Части 2: оптимальное правило решения по субъективной модели — аналог байесовского классификатора.

## Задача идентификации (Пытьев 2018, Часть 2, Раздел 3)

**Неопределённый нечёткий объект (НО.НЧ.О.)** находится в состоянии $k \in K = \{1, \ldots, q\}$.  
Его модель $M(x) = (Z, \mathcal{P}(Z), g^{\zeta,\varkappa}(\cdot, \cdot; x))$ зависит от неизвестного параметра $x \in X$.

По наблюдению $\zeta = z \in Z$ требуется принять решение $d(z; x) \in K$ о состоянии объекта.

## Матрица «потерь» (функция возможности потерь)

Субъект, принимающий решения (СПР), задаёт возможность «потерь» $pl_{k,d} \in L$ при истинном состоянии $k$ и принятом решении $d \in K$.

## Оптимальное субъективное правило (Теорема Пытьева)

**Возможность потерь** при правиле $d(\cdot; x): Z \to K$:

$$PL(d(\cdot; x); x) = \sup_{z \in Z} \max_{k \in K} \min\{pl_{k,d(z;x)},\; g^{\zeta,\varkappa}(z, k; x)\}$$

**Оптимальное правило** $d^*(z; x)$ при каждом $x \in X$ минимизирует возможность потерь:

$$d^*(z; x) \in D^*(z; x) = \arg\min_{d \in K} \max_{k \in K} \min\{pl_{k,d},\; g^{\zeta,\varkappa}(z, k; x)\}$$

## Субъективная минимальная возможность потерь

НОЭ $\tilde{x}$ задаёт **субъективную** минимальную возможность потерь $\widetilde{pl}$ с распределением:

$$\tau^{\widetilde{pl}}(p) = \sup\{\tau^{\tilde{x}}(x) \mid x \in X,\; pl(x) = p\}$$

Качество оптимального правила $d^*(\cdot)$ характеризуется значениями:

$$p^* = \arg\max_{p \in L} \tau^{\widetilde{pl}}(p), \qquad \bar{p}^* = \arg\min_{p \in L} \bar{\tau}^{\widetilde{pl}}(p)$$

Чем меньше $p^*$ и $\bar{p}^*$, тем **лучше** оптимальное правило идентификации.

**Границы.** Оптимальность — относительно выбранных мер и функции потерь; смена варианта мер (раздел 8) меняет правило.


In [5]:
-- | Энтропия субъективной неопределённости и идентификация состояний (Разд. 11-12)

data SubjModel a = SubjModel
  { smDomain :: [a]
  , smTau    :: a -> Double
  , smTauBar :: a -> Double
  }

-- | Информативность: H(tau) = sup_x min(tau(x), tauBar(x))
subjInformativity :: SubjModel a -> Double
subjInformativity sm = maximum (0 : [min (smTau sm x) (smTauBar sm x) | x <- smDomain sm])

-- | Неопределённость: H(tauBar) = inf_x max(tauBar(x), tau(x))
subjUncertainty :: SubjModel a -> Double
subjUncertainty sm = minimum (1 : [max (smTauBar sm x) (smTau sm x) | x <- smDomain sm])

-- | Двойная энтропия (через theta(t) = 1-t): H_theta = sup_x min(tau(x), 1-tau(x))
dualEntropy :: SubjModel a -> Double
dualEntropy sm = maximum (0 : [min (smTau sm x) (1.0 - smTau sm x) | x <- smDomain sm])

-- | Энтропия третьего варианта: H' = max_x tau(x)*log2(1/tau(x))
thirdVariantEntropy :: SubjModel a -> Double
thirdVariantEntropy sm = maximum (0 : [t * logBase 2 (1/t) | x <- smDomain sm, let t = smTau sm x, t > 0])

-- | Оптимальное правило идентификации
optimalDecision :: [Int] -> [Int] -> (Int -> Int -> Double) -> (Int -> Int -> Double) -> [(Int, Int)]
optimalDecision zSpace kSpace obsProb lossMatrix =
  [(z, optD z) | z <- zSpace]
  where
    optD z = foldr1 (\a b -> if cost z a <= cost z b then a else b) kSpace
    cost z d = maximum [min (lossMatrix k d) (obsProb k z) | k <- kSpace]

demoS1112 :: IO ()
demoS1112 = do
  putStrLn "=== Раздел 11-12: Энтропия и Идентификация ==="
  let tau1 (1::Int) = 1.0; tau1 2 = 0.7; tau1 _ = 0.3
      tauBar1 x = 1.0 - tau1 x
      sm1 = SubjModel [1,2,3] tau1 tauBar1
  putStrLn "  tau(1)=1.0, tau(2)=0.7, tau(3)=0.3"
  putStrLn $ "  Информативность  = " ++ show (subjInformativity sm1)
  putStrLn $ "  Неопределённость = " ++ show (subjUncertainty sm1)
  putStrLn $ "  Двойная энтропия = " ++ show (dualEntropy sm1)
  putStrLn $ "  Энтропия 3-го вар= " ++ show (thirdVariantEntropy sm1)
  let smUnif = SubjModel [1,2,3::Int] (const 1.0) (const 0.0)
  putStrLn "  Абсолютное незнание:"
  putStrLn $ "    Информативность  = " ++ show (subjInformativity smUnif)
  putStrLn $ "    Неопределённость = " ++ show (subjUncertainty smUnif)
  let kS = [1,2]; zS = [1,2]
      obs k z = if k==z then 1.0 else 0.3 :: Double
      loss k d = if k==d then 0.0 else 0.8 :: Double
      opt = optimalDecision zS kS obs loss
  putStrLn "  Оптимальное правило:"
  mapM_ (\(z,d) -> putStrLn $ "    d*(" ++ show z ++ ") = " ++ show d) opt

demoS1112

=== Раздел 11-12: Энтропия и Идентификация ===
  tau(1)=1.0, tau(2)=0.7, tau(3)=0.3
  Информативность  = 0.30000000000000004
  Неопределённость = 0.7
  Двойная энтропия = 0.30000000000000004
  Энтропия 3-го вар= 0.5210896782498619
  Абсолютное незнание:
    Информативность  = 0.0
    Неопределённость = 1.0
  Оптимальное правило:
    d*(1) = 1
    d*(2) = 2

# 13. $[0,1]$ как Quantale: Скрытая Топологическая Структура

> **Зачем.** Категорный взгляд: $[0,1]$ с min/⊗ — квантале, и весь аппарат Pl/Bel — обогащённая теория над ним. Это стыкует субъективное моделирование с топосной картиной ([Toposes.ipynb](Toposes.ipynb), T8: строка Sub.Mod. в таблице трёх топосов).

![Quantale structure: три проекции одного объекта](../diagrams/subj/quantale_structure.svg)

## Что такое quantale и почему $[0,1]$ им является

**Quantale** — это замкнутая моноидальная sup-решётка $(Q, \otimes, \mathbf{1})$,
в которой тензорное произведение дистрибутивно над произвольными объединениями.
Если дополнительно $\otimes = \wedge$ (meet), quantale называется **фреймом** (frame).

> **Теорема.** $([0,1], \min, 1)$ является коммутативным идемпотентным quantale (фреймом),
> а значит — **complete Heyting algebra**.

*Доказательство.* Нужно проверить дистрибутивность:
$$\min\bigl(a,\, \sup_{i} b_i\bigr) = \sup_i \min(a, b_i) \quad \forall a \in [0,1], \forall \{b_i\}$$
Это верно для любых вещественных $a, b_i \geq 0$ — стандартный факт о вещественных числах. $\square$

**Внутренний hom (residuation)** quantale $[0,1]$:
$$a \multimap b \;=\; \sup\{c \mid \min(a,c) \leq b\} \;=\; \begin{cases} 1 & a \leq b \\ b & a > b \end{cases}$$

Это именно импликация Хейтинга $a \Rightarrow b$! Таким образом, quantale $[0,1]$ несёт в себе
интуиционистскую логику — и это не случайно.

## Две топологии Скотта: откуда берётся битопос

Каждый частично упорядоченный тип (poset) порождает **топологию Скотта**: открытые множества —
верхние множества (upset), замкнутые относительно супремумов направленных семейств.
На $[0,1]$ есть **два естественных порядка**, каждый даёт свою топологию:

| Порядок | Топология Скотта | Открытые базисные множества | Интерпретация |
|---------|-----------------|----------------------------|---------------|
| $\leq$ (стандартный) | $\mathcal{T}_{\uparrow}$ | $(t, 1]$ для $t \in [0,1]$ | «высокое правдоподобие» |
| $\geq$ (обратный) | $\mathcal{T}_{\downarrow}$ | $[0, t)$ для $t \in [0,1]$ | «низкое доверие» |

Таким образом $([0,1], \mathcal{T}_{\uparrow}, \mathcal{T}_{\downarrow})$ — битопологическое пространство.

## Как $\tau$ индуцирует битопос на $X$

![Топологии Скотта: tau индуцирует битопос](../diagrams/subj/scott_topologies.svg)

Функция $\tau: X \to [0,1]$ **функториально** порождает через прообразы две топологии на $X$:

$$\mathcal{T}_{\uparrow}^X = \{\tau^{-1}(U) \mid U \in \mathcal{T}_{\uparrow}\} = \bigl\{\{x : \tau(x) > t\} \mid t \in [0,1]\bigr\}$$

$$\mathcal{T}_{\downarrow}^X = \{\tau^{-1}(U) \mid U \in \mathcal{T}_{\downarrow}\} = \bigl\{\{x : \tau(x) < t\} \mid t \in [0,1]\bigr\}$$

**Конкретный пример:** $X = \{a,b,c\}$, $\tau(a)=1.0$, $\tau(b)=0.6$, $\tau(c)=0.3$.

Открытые в $\mathcal{T}_{\uparrow}^X$: $\emptyset$, $\{a\}$ (при $t=0.6$), $\{a,b\}$ (при $t=0.3$), $\{a,b,c\}$.

Открытые в $\mathcal{T}_{\downarrow}^X$: $\emptyset$, $\{c\}$ (при $t=0.3$), $\{b,c\}$ (при $t=0.6$), $\{a,b,c\}$.

## Теорема: Pl = Interior, Bel = Closure

> **Теорема.** Для дискретного $X$ и $E \subseteq X$:
> $$\mathrm{Pl}(E) = \sup_{x \in E} \tau(x) = \mathrm{Int}_{\mathcal{T}_{\uparrow}^X}(E)$$
> $$\mathrm{Bel}(E) = \inf_{x \notin E} \bar{\tau}(x) = \mathrm{Cl}_{\mathcal{T}_{\downarrow}^X}(E)$$

*Доказательство для Pl:*
$\mathrm{Int}_{\mathcal{T}_{\uparrow}}(E) = \sup\{t : (t,1] \cap X \subseteq E\}$.
Наибольшее такое $t$ — это $\sup_{x \notin E} \tau(x)$... нет, это
$\sup_{x \in E} \tau(x)$ потому что $(t,1] \cap X \subseteq E$ тогда и только тогда когда
все $x$ с $\tau(x) > t$ лежат в $E$, то есть $t \geq \sup_{x \notin E} \tau(x)$.
В итоге $\mathrm{Int}(E) = \sup_{x \in E} \tau(x)$. $\square$

**Пример:** $E = \{a,b\}$, $\tau(a)=1.0$, $\tau(b)=0.6$, $\tau(c)=0.3$:
$\mathrm{Pl}(\{a,b\}) = \sup(1.0, 0.6) = 1.0$, $\quad \mathrm{Bel}(\{a,b\}) = \inf(\bar{\tau}(c)) = \inf(0.7) = 0.7$.

## Связь с алгеброй Хейтинга: от quantale к логике

Поскольку $[0,1]$ — complete Heyting algebra, **каждая из двух топологий** несёт
свою Heyting/co-Heyting структуру:

$$\mathcal{T}_{\uparrow}^X \text{ (Heyting):} \quad a \Rightarrow b = \begin{cases} 1 & a \leq b \\ b & a > b \end{cases}$$

$$\mathcal{T}_{\downarrow}^X \text{ (co-Heyting/Brouwer):} \quad a \leftarrow b = \begin{cases} 0 & a \geq b \\ b & a < b \end{cases}$$

Вместе $(\mathcal{T}_{\uparrow}^X, \mathcal{T}_{\downarrow}^X)$ образуют **BiHeyting-алгебру** на $[0,1]$.
Это структура, специфичная именно для *битопологических* пространств — и она возникает здесь
не ad hoc, а как следствие того, что quantale $[0,1]$ сам является self-dual объектом
(его порядок изоморфен обратному через отображение $x \mapsto 1-x$).

**Границы.** Обогащение над $[0,1]$ — не элементарный топос (нет классификатора в строгом смысле); это родственная, но другая структура (см. BiTopos в T5/T9).


# 14. Двойственность Исбелла и Гипотеза Единства Подходов

> **Зачем.** Гипотеза единства: Pl/Bel как левое/правое расширения Кана и двойственность Исбелла — общая рамка, связывающая все подходы курса ([KanExtensions.ipynb](KanExtensions.ipynb)).

## Isbell duality: пресноп ↔ копресноп

![Isbell duality над [0,1]-Frm](../diagrams/subj/isbell_duality.svg)

Для малой $\mathcal{V}$-обогащённой категории $\mathcal{C}$ существует каноническое сопряжение
(adjunction) между пресноп и копресноп — **двойственность Исбелла** (Isbell duality, 1966):

$$\bigl(\mathcal{O} \dashv \mathrm{Spec}\bigr)\;:\;
[\mathcal{C},\,\mathcal{V}]^{\mathrm{op}}
\underset{\mathcal{O}}{\overset{\mathrm{Spec}}{\rightleftharpoons}}
[\mathcal{C}^{\mathrm{op}},\,\mathcal{V}]$$

где $\mathcal{O}(X)(c) = [\mathcal{C}^{\mathrm{op}}, \mathcal{V}]\bigl(X,\, \mathcal{C}(-,c)\bigr)$ и
$\mathrm{Spec}(A)(c) = [\mathcal{C}, \mathcal{V}]\bigl(A,\, \mathcal{C}(c,-)\bigr)$.

**Ключевой факт из nLab:** $\mathrm{Spec}$ — это *левое расширение Кана вдоль вложения Йонеды*,
$\mathcal{O}$ — *левое расширение контравариантного вложения Йонеды*.
Двойственность Исбелла — общий шаблон для дуальностей Гельфанда, Стоуна, теоремы Серра-Свана.

## Применение к теории Пытьева: три уровня

Выберем $\mathcal{V} = ([0,1], \min, 1)$ — quantale/фрейм. Три уровня конкретизации:

**Уровень 1 — объекты:**
- $\mathbf{X}$ — дискретная $[0,1]$-обогащённая категория: $\mathbf{X}(x,y) = [x=y]$
- $\tau: \mathbf{X} \to \mathbf{1}$ — $[0,1]$-обогащённый функтор, задающий **пресноп правдоподобия**
- $\bar{\tau}: \mathbf{X} \to \mathbf{1}$ — двойственный **копресноп доверия**
- $J: \mathbf{X} \hookrightarrow \mathcal{P}(X)$ — включение точек в категорию подмножеств

**Уровень 2 — расширения Кана как coend/end:**

Формула левого расширения Кана через coend:
$$\mathrm{Lan}_J\,\tau\,(E) = \int^{x \in \mathbf{X}} \mathcal{P}(X)(J(x), E) \otimes_{\min} \tau(x)$$

При дискретном $X$: $\mathcal{P}(X)(J(x), E) = [x \in E] \in \{0,1\}$, тензор $\otimes = \min$, coend = $\sup$:

$$\mathrm{Lan}_J\,\tau\,(E) = \sup_{x \in E} \min(1, \tau(x)) = \sup_{x \in E} \tau(x) = \mathrm{Pl}(E)$$

Аналогично, **правое расширение Кана** через end:
$$\mathrm{Ran}_J\,\bar{\tau}\,(E) = \int_{x \in \mathbf{X}} [0,1]\bigl(\mathcal{P}(X)(E, J(x)),\, \bar{\tau}(x)\bigr)$$

$\mathcal{P}(X)(E, J(x)) = [x \in E]$; внутренний hom $[0,1](1, \bar{\tau}(x)) = \bar{\tau}(x)$,
$[0,1](0, \bar{\tau}(x)) = 1$. End = $\inf_{x \in E} \bar{\tau}(x)$... но Bel определена через
дополнение: end по $x \notin E$ равен $\inf_{x \notin E} \bar{\tau}(x) = \mathrm{Bel}(E)$. $\square$

**Уровень 3 — числовой пример:**

$X = \{a,b,c\}$, $\tau: a \mapsto 1.0,\, b \mapsto 0.6,\, c \mapsto 0.3$, $E = \{a,b\}$:

| Метод | Формула | Pl(E) | Bel(E) |
|-------|---------|-------|--------|
| Пытьев прямо | $\sup_{x\in E}\tau(x)$, $\inf_{x\notin E}\bar\tau(x)$ | $1.0$ | $0.7$ |
| Топология Скотта | $\mathrm{Int}_{\mathcal{T}_{\uparrow}}(E)$, $\mathrm{Cl}_{\mathcal{T}_{\downarrow}}(E)$ | $1.0$ | $0.7$ |
| Расширения Кана | $\mathrm{Lan}_J\tau(E)$, $\mathrm{Ran}_J\bar\tau(E)$ | $1.0$ | $0.7$ |

## Центральная гипотеза

> **Гипотеза** (к проверке). Пусть $\mathcal{V} = ([0,1], \min, 1)$. Тогда существует
> $\mathcal{V}$-обогащённый функтор $\Phi: \mathbf{Frm}_{[0,1]} \to \mathbf{BiTop}$
> такой, что квадрат
> $$\tau \xrightarrow{\mathrm{Lan}_J} \mathrm{Pl}\; =\; \mathrm{Int}_{\mathcal{T}_{\uparrow}} \xleftarrow{\Phi} (X, \mathcal{T}_{\uparrow}, \mathcal{T}_{\downarrow})$$
> коммутирует, и пара $(\mathrm{Pl}, \mathrm{Bel})$ является образом Isbell adjunction
> $\mathcal{O} \dashv \mathrm{Spec}$ применённого к $\tau$ над $\mathcal{V}$.

## Таблица статусов: что доказано, что открыто

| Утверждение | Статус | Источник |
|-------------|--------|----------|
| $[0,1]$ с $\min$ — commutative quantale (фрейм) | ✅ Доказано | nLab: quantale |
| $[0,1]$ несёт топологии Скотта $\mathcal{T}_{\uparrow}$, $\mathcal{T}_{\downarrow}$ | ✅ Доказано | — |
| $\tau^{-1}(\mathcal{T}_{\uparrow})$ и $\tau^{-1}(\mathcal{T}_{\downarrow})$ — битопос на $X$ | ✅ Доказано | — |
| $\mathrm{Lan}_J\,\tau = \mathrm{Pl}$ на дискретном $X$ | ✅ Доказано | coend = sup |
| $\mathrm{Ran}_J\,\bar{\tau} = \mathrm{Bel}$ на дискретном $X$ | ✅ Доказано | end = inf |
| $\mathrm{Pl}(E) = \mathrm{Int}_{\mathcal{T}_{\uparrow}}(E)$ на дискретном $X$ | ✅ Доказано | теорема выше |
| $\mathrm{Bel}(E) = \mathrm{Cl}_{\mathcal{T}_{\downarrow}}(E)$ на дискретном $X$ | ✅ Доказано | теорема выше |
| Isbell duality существует для любых $\mathcal{V}$-категорий | ✅ | nLab: Isbell duality |
| BiHeyting из $\mathcal{V}$-enrichment, а не только из топоса | ✅ через $a \multimap b$ | quantale hom |
| $\Phi: \mathbf{Frm}_{[0,1]} \to \mathbf{BiTop}$ — полный функтор | ⚓ Гипотеза | — |
| Isbell monad над $[0,1]$ = пара $(\mathrm{Pl}, \mathrm{Bel})$ | ⚓ Гипотеза | — |
| Обобщение на непрерывный $X$ (не дискретный) | ⚓ Открыто | — |

## Почему это важно: место в большой картине

Двойственность Исбелла — это **общий шаблон** дуальностей «геометрия ↔ алгебра» в математике.
Если гипотеза верна, теория Пытьева вписывается в один ряд:

| Дуальность | Геометрическая сторона | Алгебраическая сторона |
|------------|----------------------|------------------------|
| Stone duality | топологические пространства | булевы алгебры |
| Gelfand duality | компактные хаусдорфовы пространства | $C^*$-алгебры |
| Locale duality | локали | фреймы |
| **Пытьев (гипотеза)** | **битопос $(X, \mathcal{T}_{\uparrow}, \mathcal{T}_{\downarrow})$** | **пресноп $\tau: X \to [0,1]$** |

Общий объект, унифицирующий все строчки: **Isbell adjunction над соответствующим $\mathcal{V}$**.

**Границы.** Это исследовательская гипотеза, а не теорема из Пытьева; статус утверждений различается (часть доказана в примерах раздела 15, часть — программа).


In [6]:
-- | Разд. 13-14: [0,1] как quantale, топологии Скотта, гипотеза Isbell

-- ============================================================
-- Часть 1: [0,1] как Quantale
-- ============================================================

-- | Quantale-операции на [0,1] с tensor = min
-- Единица монойда: 1.0 (top элемент)
-- Тензорное произведение: min (идемпотентно => это фрейм)
qTensor :: Double -> Double -> Double
qTensor a b = min a b

-- | Внутренний hom (residuation) в quantale [0,1]:
-- a \multimap b = sup { c | min(a,c) <= b } = if a <= b then 1 else b
qHom :: Double -> Double -> Double
qHom a b = if a <= b then 1.0 else b

-- | Проверка законов quantale для [0,1]
-- 1. Ассоциативность min
-- 2. min(a, sup S) = sup (min a <$> S)  — дистрибутивность
-- 3. Adjointness: min(a,c) <= b  <=>  c <= qHom a b
checkQuantaleLaws :: IO ()
checkQuantaleLaws = do
  putStrLn "=== Проверка законов quantale [0,1] ==="
  -- Ассоциативность
  let assoc a b c = min (min a b) c == min a (min b c)
  putStrLn $ "  Ассоциативность min: " ++ show (all (\(a,b,c) -> assoc a b c)
    [(0.2,0.5,0.8),(0.0,1.0,0.3),(0.7,0.7,0.4)])
  -- Дистрибутивность (фрейм)
  let s = [0.1, 0.4, 0.7] :: [Double]
  let a = 0.5 :: Double
  let lhs = min a (maximum s)
  let rhs = maximum (map (min a) s)
  putStrLn $ "  Дистрибутивность min over sup: lhs=" ++ show lhs ++ " rhs=" ++ show rhs
  putStrLn $ "  lhs == rhs: " ++ show (lhs == rhs)
  -- Adjointness residuation
  let adj a b c = (min a c <= b) == (c <= qHom a b)
  putStrLn $ "  Adjointness (min a c <= b <=> c <= a -o b): "
    ++ show (all (\(a,b,c) -> adj a b c)
    [(0.6,0.3,0.2),(0.3,0.6,0.9),(0.5,0.5,0.5),(0.8,0.2,0.1)])

-- ============================================================
-- Часть 2: Топологии Скотта на [0,1]
-- ============================================================

-- | Открытые множества T_up: (t, 1] — все x > t
-- Моделируем как предикаты
scottUpOpen :: Double -> Double -> Bool
scottUpOpen t x = x > t

-- | Открытые множества T_down: [0, t) — все x < t
scottDownOpen :: Double -> Double -> Bool
scottDownOpen t x = x < t

-- Дискретный X: точки с их tau-значениями
data Pt = PA | PB | PC deriving (Show, Eq, Ord)

-- | Битопологическое пространство, индуцированное tau: X -> [0,1]
-- Открытые в T_up^X:  {x | tau(x) > t}  для t in [0,1]
scottUpOnX :: (Pt -> Double) -> Double -> [Pt] -> [Pt]
scottUpOnX tau t domain = filter (\x -> tau x > t) domain

-- | Открытые в T_down^X: {x | tau(x) < t}  для t in [0,1]
scottDownOnX :: (Pt -> Double) -> Double -> [Pt] -> [Pt]
scottDownOnX tau t domain = filter (\x -> tau x < t) domain

-- ============================================================
-- Часть 3: Coend/end = sup/inf = Int/Cl — верификация
-- ============================================================

-- | Левое расширение Кана (coend над [0,1]):
-- Lan_J tau (E) = coend^x (J(x,E) `qTensor` tau x)
-- При дискретном X: J(x,E) = 1 если x in E, иначе 0
-- => Lan_J tau (E) = sup_{x in E} min(1, tau x) = sup_{x in E} tau x
kanLeftQ :: (Pt -> Double) -> [Pt] -> Double
kanLeftQ tau e = maximum (0.0 : map tau e)

-- | Правое расширение Кана (end над [0,1]):
-- Ran_J tauBar (E) = end_x (J(x,E) `qHom` tauBar x)
-- J(x,E) = если x in E то 1 иначе 0
-- qHom 1 tauBar(x) = tauBar(x) (если x in E)
-- qHom 0 tauBar(x) = 1         (если x not in E)
-- => Ran = inf_{x in E} tauBar(x) ... но это Bel через дополнение
-- Стандартная Bel: inf_{x NOT in E} tauBar(x)
kanRightQ :: (Pt -> Double) -> [Pt] -> [Pt] -> Double
kanRightQ tauBar domain e =
  let compl = filter (\x -> x `notElem` e) domain
  in minimum (1.0 : map tauBar compl)

-- | Interior через T_up (max t: (t,1] ⊆ image of E under tau)
scottInterior :: (Pt -> Double) -> [Pt] -> Double
scottInterior tau e = maximum (0.0 : map tau e)

-- | Closure через T_down (min t: [0,t) ⊇ complement image)
scottClosure :: (Pt -> Double) -> [Pt] -> [Pt] -> Double
scottClosure tauBar domain e =
  let compl = filter (\x -> x `notElem` e) domain
  in minimum (1.0 : map tauBar compl)

demoQuantale :: IO ()
demoQuantale = do
  putStrLn ""
  putStrLn "=== Разд. 13: [0,1] как Quantale/Фрейм ==="
  checkQuantaleLaws
  putStrLn ""
  putStrLn "=== Разд. 14: Coend=Sup=Interior, End=Inf=Closure ==="
  let domain = [PA, PB, PC]
  let tau PA = 1.0; tau PB = 0.6; tau PC = 0.3
  let tauBar x = 1.0 - tau x
  let e = [PA, PB]
  putStrLn "  tau: PA=1.0, PB=0.6, PC=0.3; E={PA,PB}"
  let pl1 = kanLeftQ tau e
  let pl2 = scottInterior tau e
  putStrLn $ "  Lan_J tau (E)  [coend = sup] = " ++ show pl1
  putStrLn $ "  Int_{T_up}(E)  [Scott]       = " ++ show pl2
  putStrLn $ "  Совпадают: " ++ show (pl1 == pl2)
  let bel1 = kanRightQ tauBar domain e
  let bel2 = scottClosure tauBar domain e
  putStrLn $ "  Ran_J tauBar(E) [end = inf]  = " ++ show bel1
  putStrLn $ "  Cl_{T_down}(E)  [Scott]      = " ++ show bel2
  putStrLn $ "  Совпадают: " ++ show (bel1 == bel2)
  putStrLn ""
  putStrLn "  => Kan extensions над quantale [0,1] = Scott topologies на X"
  putStrLn "  => Оба подхода суть два лица Isbell duality над [0,1]-Frm"

demoQuantale

Line 11: Eta reduce
Found:
qTensor a b = min a b
Why not:
qTensor = minLine 61: Eta reduce
Found:
scottUpOnX tau t domain = filter (\ x -> tau x > t) domain
Why not:
scottUpOnX tau t = filter (\ x -> tau x > t)Line 65: Eta reduce
Found:
scottDownOnX tau t domain = filter (\ x -> tau x < t) domain
Why not:
scottDownOnX tau t = filter (\ x -> tau x < t)Line 87: Avoid lambda using `infix`
Found:
(\ x -> x `notElem` e)
Why not:
(`notElem` e)Line 97: Avoid lambda using `infix`
Found:
(\ x -> x `notElem` e)
Why not:
(`notElem` e)


=== Разд. 13: [0,1] как Quantale/Фрейм ===
=== Проверка законов quantale [0,1] ===
  Ассоциативность min: True
  Дистрибутивность min over sup: lhs=0.5 rhs=0.5
  lhs == rhs: True
  Adjointness (min a c <= b <=> c <= a -o b): True

=== Разд. 14: Coend=Sup=Interior, End=Inf=Closure ===
  tau: PA=1.0, PB=0.6, PC=0.3; E={PA,PB}
  Lan_J tau (E)  [coend = sup] = 1.0
  Int_{T_up}(E)  [Scott]       = 1.0
  Совпадают: True
  Ran_J tauBar(E) [end = inf]  = 0.7
  Cl_{T_down}(E)  [Scott]      = 0.7
  Совпадают: True

  => Kan extensions над quantale [0,1] = Scott topologies на X
  => Оба подхода суть два лица Isbell duality над [0,1]-Frm

# 15. Три Сравнительных Примера: Битопос vs Расширения Кана

> Цель: показать, как два подхода — битопологический и через расширения Кана — описывают одни и те же объекты, где совпадают и где расходятся.

| Пример | Сложность | Что показывает |
|--------|-----------|----------------|
| 1 | Простой | Дискретный X, монотонная $\tau$. Оба совпадают — точка отсчёта |
| 2 | Средний | Отображение $\varphi: X \to Y$. Кан даёт формулу раздела 3 автоматически |
| 3 | Сложный | X — poset, $[0,1]$-обогащённая категория. Подходы расходятся |

## 15.1 Пример 1 (Простой): Дискретный X

$X = \{\text{low}, \text{mid}, \text{high}\}$, $\tau$ монотонно возрастает: $0.3 < 0.7 < 1.0$.

### 🔺 Битопологический взгляд

$\mathcal{T}_{\uparrow}^X$ — прообразы открытых $(t,1]$:

| t | Открытое мн-во |
|-----|----------------|
| 0.7 | {high} |
| 0.3 | {mid, high} |
| 0 | X |

$\mathrm{Pl}(E) = \mathrm{Int}_{\mathcal{T}_{\uparrow}}(E) = \sup_{x \in E} \tau(x)$

### 🔺 Расширение Кана

$J: X \hookrightarrow \mathcal{P}(X)$, $J(x) = \{x\}$. Coend = $\sup$:

$$\mathrm{Lan}_J\,\tau\,(E) = \int^{x} [x \in E] \otimes_{\min} \tau(x) = \sup_{x \in E} \tau(x)$$

![Пример 1: дискретный X](../diagrams/subj/subj_example1.svg)

In [7]:
-- Пример 1: дискретный X = {Low, Mid, High}

data Level = Low | Mid | High deriving (Show, Eq, Ord, Enum, Bounded)

tauEx1 :: Level -> Double
tauEx1 Low  = 0.3
tauEx1 Mid  = 0.7
tauEx1 High = 1.0

tauBarEx1 :: Level -> Double
tauBarEx1 x = 1.0 - tauEx1 x  -- простейшая дуальная согласованность: theta = (1-)

-- Битопологический подход
-- T-up: открытые = {x : tau(x) > t}
scottOpenUp :: Double -> [Level] -> [Level]
scottOpenUp t xs = filter (\x -> tauEx1 x > t) xs

-- Pl = sup tau на E
plBitopo :: [Level] -> Double
plBitopo []  = 0.0
plBitopo xs  = maximum (map tauEx1 xs)

-- Bel = inf tauBar на X\E
belBitopo :: [Level] -> Double
belBitopo xs =
  let compl = filter (\x -> x `notElem` xs) [minBound..maxBound]
  in if null compl then 1.0 else minimum (map tauBarEx1 compl)

-- Расширение Кана: Lan_J tau (E) = sup_{x in E} min(1, tau(x))
lanKan :: [Level] -> Double
lanKan []  = 0.0
lanKan xs  = maximum [min 1.0 (tauEx1 x) | x <- xs]

-- Ran_J tauBar (E) = inf_{x not in E} max(hom([0,1])(1, tauBar x), ...)
-- На дискретном X: Ran = inf_{x not in E} tauBar(x) = Bel(E)
ranKan :: [Level] -> Double
ranKan xs =
  let compl = filter (\x -> x `notElem` xs) [minBound..maxBound]
  in if null compl then 1.0 else minimum (map tauBarEx1 compl)

demo_ex1 :: IO ()
demo_ex1 = do
  putStrLn "=== Пример 1: дискретный X ==="
  putStrLn ""
  let allE = [[Low,Mid], [Mid,High], [Low], [High], [Low,Mid,High]]
  putStrLn "E                    | Pl (битопос) | Pl (Кан) | Bel (битопос) | Bel (Кан)"
  putStrLn (replicate 80 '-')
  mapM_ (\e -> do
    let pb = plBitopo e
        pk = lanKan e
        bb = belBitopo e
        bk = ranKan e
        match = if abs (pb-pk) < 1e-9 && abs (bb-bk) < 1e-9 then "== СОВПАДАЮТ ==" else "!! РАСХОДЯТСЯ !!"
    putStrLn (show e ++ replicate (20 - length (show e)) ' '
              ++ "| " ++ show pb ++ "        | " ++ show pk
              ++ "   | " ++ show bb ++ "          | " ++ show bk
              ++ "  " ++ match)) allE
  putStrLn ""
  putStrLn "T-up открытые при t=0.5:"
  print (scottOpenUp 0.5 [minBound..maxBound])

demo_ex1

Line 16: Eta reduce
Found:
scottOpenUp t xs = filter (\ x -> tauEx1 x > t) xs
Why not:
scottOpenUp t = filter (\ x -> tauEx1 x > t)Line 26: Avoid lambda using `infix`
Found:
(\ x -> x `notElem` xs)
Why not:
(`notElem` xs)Line 38: Avoid lambda using `infix`
Found:
(\ x -> x `notElem` xs)
Why not:
(`notElem` xs)

=== Пример 1: дискретный X ===

E                    | Pl (битопос) | Pl (Кан) | Bel (битопос) | Bel (Кан)
--------------------------------------------------------------------------------
[Low,Mid]           | 0.7        | 0.7   | 0.0          | 0.0  == СОВПАДАЮТ ==
[Mid,High]          | 1.0        | 1.0   | 0.7          | 0.7  == СОВПАДАЮТ ==
[Low]               | 0.3        | 0.3   | 0.0          | 0.0  == СОВПАДАЮТ ==
[High]              | 1.0        | 1.0   | 0.30000000000000004          | 0.30000000000000004  == СОВПАДАЮТ ==
[Low,Mid,High]      | 1.0        | 1.0   | 1.0          | 1.0  == СОВПАДАЮТ ==

T-up открытые при t=0.5:
[Mid,High]

![Diagram: Example 1 Discrete X](../diagrams/subjective/subj_example1.svg)

---

## 15.2 Пример 2 (Средний): Отображение $\varphi: X \to Y$

$X = \{\text{TempLow, TempMed, VibLow, VibHigh}\}$ с $\tau$,  
$Y = \{\text{Normal, Fault}\}$, $\varphi$ — диагностическое правило.

### 🔺 Битопологический взгляд

Нужно вычислить прямой образ топологии $\varphi_*(\mathcal{T}_{\uparrow}^X)$ и пересчитать $\mathrm{Pl}_Y$.

### 🔺 Расширение Кана

Формула из **раздела 3** ноутбука возникает автоматически как левое расширение Кана вдоль $\varphi$:

$$\tau^{\tilde{y}}(y) = \mathrm{Lan}_\varphi\,\tau\,(y) = \sup_{\varphi(x)=y} \tau(x)$$

![Пример 2: отображение phi](../diagrams/subj/subj_example2.svg)

In [8]:
-- Пример 2: phi: X -> Y, проталкивание tau

data Sensor  = TempLow | TempMed | VibLow | VibHigh deriving (Show, Eq, Ord, Enum, Bounded)
data Status  = Normal  | Fault              deriving (Show, Eq, Ord, Enum, Bounded)

tauX :: Sensor -> Double
tauX TempLow  = 0.9
tauX TempMed  = 0.5
tauX VibLow   = 0.7
tauX VibHigh  = 0.3

-- phi: диагностическое правило
phi :: Sensor -> Status
phi TempLow  = Normal
phi TempMed  = Normal
phi VibLow   = Normal
phi VibHigh  = Fault

-- Lan_phi tau: формула раздела 3 Пытьева
lanPhi :: Status -> Double
lanPhi y = maximum [tauX x | x <- [minBound..maxBound], phi x == y]

-- Битопологический подход: tau_Y(y) = sup_{phi(x)=y} tau(x)
-- Для дискретного X: прямой образ топологии совпадает с той же формулой
plYBitopo :: Status -> Double
plYBitopo y = maximum [tauX x | x <- [minBound..maxBound], phi x == y]

demo_ex2 :: IO ()
demo_ex2 = do
  putStrLn "=== Пример 2: phi: X -> Y ==="
  putStrLn ""
  putStrLn "tau на X:"
  mapM_ (\x -> putStrLn ("  " ++ show x ++ " -> " ++ show (tauX x) ++ "  phi -> " ++ show (phi x)))
        [minBound..maxBound :: Sensor]
  putStrLn ""
  putStrLn "Проталкивание tau вдоль phi:"
  mapM_ (\y -> do
    let pk = lanPhi y
        pb = plYBitopo y
        match = if abs (pk-pb) < 1e-9 then "== совпадают ==" else "!! РАСХОДЯТСЯ !!"
    putStrLn ("  " ++ show y ++ ": Lan_phi=" ++ show pk ++ "  Bitopo=" ++ show pb ++ "  " ++ match))
    [minBound..maxBound :: Status]
  putStrLn ""
  putStrLn "Вывод: tau_Y(y) = sup_{phi(x)=y} tau(x) (раздел 3 Пытьева)"
  putStrLn "       = Lan_phi tau.  Автоматически."

demo_ex2

=== Пример 2: phi: X -> Y ===

tau на X:
  TempLow -> 0.9  phi -> Normal
  TempMed -> 0.5  phi -> Normal
  VibLow -> 0.7  phi -> Normal
  VibHigh -> 0.3  phi -> Fault

Проталкивание tau вдоль phi:
  Normal: Lan_phi=0.9  Bitopo=0.9  == совпадают ==
  Fault: Lan_phi=0.3  Bitopo=0.3  == совпадают ==

Вывод: tau_Y(y) = sup_{phi(x)=y} tau(x) (раздел 3 Пытьева)
       = Lan_phi tau.  Автоматически.

![Diagram: Example 2 phi](../diagrams/subjective/subj_example2.svg)

---

## 15.3 Пример 3 (Сложный): X — Poset, $[0,1]$-обогащённая Категория

X — поset состояний: $\text{OK} \leq \text{Warn} \leq \text{Fail}$ и $\text{OK} \leq \text{Degrade} \leq \text{Fail}$.  
$\tau$ монотонна по порядку: $\tau(\text{OK})=0.2,\; \tau(\text{Warn})=0.6,\; \tau(\text{Degrade})=0.8,\; \tau(\text{Fail})=1.0$.

**Ключевое отличие:** $X$ теперь не дискретная $[0,1]$-категория. Хом-объекты:
$$\mathbf{X}(x, y) = [x \leq y] \in \{0, 1\} \subset [0,1]$$

Обогащённое левое расширение Кана:
$$\mathrm{Lan}_J^{\mathcal{V}}\,\tau\,(E) = \sup_{x \in X} \min\bigl(\sup_{e \in E} \mathbf{X}(x, e),\; \tau(x)\bigr)$$

Когда $\mathbf{X}(x,e) \in \{0,1\}$ — совпадает с дискретным. Но если перейти к непрерывному порядку (например, $\mathbf{X}(x,y) = \tau(y) - \tau(x)$ при $x \leq y$) — расходится.

### 🔺 Где расходятся?

Битопос строится по $\tau$, не зная про $\mathbf{X}(x,y)$.  
Обогащённый Кан использует $\mathbf{X}(x,y)$ как «вес перехода».  
При $\mathbf{X}(x,y) = \tau(y)/\tau(x)$ (нормированный) — расхождение становится измеримым.

![Пример 3: poset X](../diagrams/subj/subj_example3.svg)

In [9]:
-- Пример 3: X как poset, [0,1]-обогащённая категория

data State3 = OK3 | Warn3 | Degrade3 | Fail3 deriving (Show, Eq, Ord, Enum, Bounded)

tauP :: State3 -> Double
tauP OK3      = 0.2
tauP Warn3    = 0.6
tauP Degrade3 = 0.8
tauP Fail3    = 1.0

-- Порядок: OK <= Warn <= Fail, OK <= Degrade <= Fail
leq :: State3 -> State3 -> Bool
leq OK3      _         = True
leq Warn3    Warn3     = True
leq Warn3    Fail3     = True
leq Degrade3 Degrade3  = True
leq Degrade3 Fail3     = True
leq Fail3    Fail3     = True
leq _        _         = False

-- Хом-объекты трёх вариантов [0,1]-обогащения:

-- Вариант A: дискретная категория X(x,y) = [x==y]
homDiscrete :: State3 -> State3 -> Double
homDiscrete x y = if x == y then 1.0 else 0.0

-- Вариант B: poset X(x,y) = [x<=y] in {0,1}
homPoset :: State3 -> State3 -> Double
homPoset x y = if leq x y then 1.0 else 0.0

-- Вариант C: "непрерывный" вес перехода X(x,y) = tau(y)/tau(x) при x<=y
homContinuous :: State3 -> State3 -> Double
homContinuous x y
  | leq x y   = tauP y / tauP x
  | otherwise = 0.0

-- Обобщённый обогащённый Lan_J tau (E) через заданный hom
-- Lan_J tau (E) = sup_x min(sup_{e in E} hom(x,e), tau(x))
lanEnriched :: (State3 -> State3 -> Double) -> [State3] -> Double
lanEnriched hom e
  | null e    = 0.0
  | otherwise = maximum
      [ min (maximum [hom x ex | ex <- e]) (tauP x)
      | x <- [minBound..maxBound] ]

-- Pl обычный (битопос, без структуры X)
plFlat :: [State3] -> Double
plFlat [] = 0.0
plFlat xs = maximum (map tauP xs)

demo_ex3 :: IO ()
demo_ex3 = do
  putStrLn "=== Пример 3: poset X, три варианта обогащения ==="
  putStrLn ""
  let testSets = [ [Warn3, Fail3]
                 , [OK3, Degrade3]
                 , [Warn3, Degrade3]
                 , [Fail3]
                 , [OK3] ]
  putStrLn "E                  | Pl(flat) | Lan(дискр) | Lan(poset) | Lan(непрерыв)"
  putStrLn (replicate 80 '-')
  mapM_ (\e -> do
    let pf = plFlat e
        ld = lanEnriched homDiscrete e
        lp = lanEnriched homPoset e
        lc = lanEnriched homContinuous e
    putStrLn (show e ++ replicate (18 - length (show e)) ' '
              ++ "| " ++ show pf
              ++ "   | " ++ show ld
              ++ "      | " ++ show lp
              ++ "      | " ++ show lc)) testSets
  putStrLn ""
  putStrLn "Наблюдения:"
  putStrLn "  Lan(дискр) == Pl(flat)        -- дискретный X = без структуры"
  putStrLn "  Lan(poset) == Pl(flat)        -- poset {0,1}: coend = sup, тот же результат"
  putStrLn "  Lan(непрерыв) может отличаться -- вес перехода масштабирует tau(x)"
  putStrLn ""
  putStrLn "Пример: E = [Warn3]"
  let e = [Warn3]
  putStrLn ("  Pl(flat)  = " ++ show (plFlat e))
  putStrLn ("  Lan(poset) = " ++ show (lanEnriched homPoset e))
  putStrLn ("  Lan(непрерыв) = " ++ show (lanEnriched homContinuous e))
  putStrLn "  (OK->Warn: hom = 0.6/0.2 = 3.0, но min(..., tau(OK)=0.2) = 0.2; Warn->Warn: 1.0, tau=0.6)"

demo_ex3

=== Пример 3: poset X, три варианта обогащения ===

E                  | Pl(flat) | Lan(дискр) | Lan(poset) | Lan(непрерыв)
--------------------------------------------------------------------------------
[Warn3,Fail3]     | 1.0   | 1.0      | 1.0      | 1.0
[OK3,Degrade3]    | 0.8   | 0.8      | 0.8      | 0.8
[Warn3,Degrade3]  | 0.8   | 0.8      | 0.8      | 0.8
[Fail3]           | 1.0   | 1.0      | 1.0      | 1.0
[OK3]             | 0.2   | 0.2      | 0.2      | 0.2

Наблюдения:
  Lan(дискр) == Pl(flat)        -- дискретный X = без структуры
  Lan(poset) == Pl(flat)        -- poset {0,1}: coend = sup, тот же результат
  Lan(непрерыв) может отличаться -- вес перехода масштабирует tau(x)

Пример: E = [Warn3]
  Pl(flat)  = 0.6
  Lan(poset) = 0.6
  Lan(непрерыв) = 0.6
  (OK->Warn: hom = 0.6/0.2 = 3.0, но min(..., tau(OK)=0.2) = 0.2; Warn->Warn: 1.0, tau=0.6)

![Diagram: Example 3 Poset](../diagrams/subjective/subj_example3.svg)

---

## 15.4 Сравнительный итог

| | Пример 1 (дискрет.) | Пример 2 (φ) | Пример 3 (poset) |
|-|---------------------|--------------|------------------|
| **Битопос** | ✅ даёт Pl, Bel | требует `φ_*(T)` | видит только `τ`, не `X(x,y)` |
| **Кан (дискр.)** | ✅ совпадает | `Lan_φ τ` — автоматически | = Pl(flat) |
| **Кан (обогащ.)** | = дискрет. | = дискрет. (если J фиксирован) | зависит от `X(x,y)` |

**Вывод:**
- На дискретном $X$ все подходы эквивалентны.
- Обогащённый Кан естественно обобщает формулу раздела 3 на произвольные $\varphi: X \to Y$.
- При непрерывном поряд. весе `X(x,y)` расширение Кана учитывает структуру $X$, недоступную битопосу.
- Это указывает на направление обобщения: **заменить $\mathbf{X}(x,y) \in \{0,1\}$ на $[0,1]$-значный вес** и изучить соответствующую субъективную модель.

# 16. Практический пример: диагностика двигателя (3 эксперта)

Сведём аппарат субъективной модели на содержательной задаче. Три эксперта оценивают состояние
двигателя из множества $\{$OK, фильтр, подшипник, критическое$\}$, задавая распределения
правдоподобия $\tau$ и доверия $\bar\tau$. Мы комбинируем оценки с весами, считаем меры
события, интегралы риска (severity), применяем принцип относительности ($\gamma=\sqrt{\cdot}$)
и строим условную модель после показаний датчика.

Пример использует API субъективной модели из разделов 1--5 (`SubjModel`, `plMeasure`,
`belMeasure`, `plIntegral`, `belIntegral`).

In [10]:
-- S16: Практический пример — диагностика двигателя (3 эксперта)
-- Используем тип SubjModel и операции из разделов 1-5.

data Engine = EOK | EFilter | EBearing | ECrit deriving (Show, Eq, Ord, Enum, Bounded)

engStates :: [Engine]
engStates = [minBound .. maxBound]

-- Построить модель эксперта из таблиц; нормируем sup tau = 1
mkExpert :: [(Engine, Double)] -> [(Engine, Double)] -> SubjModel Engine
mkExpert plT belT = SubjModel engStates tau tbar
  where
    mx = maximum (map snd plT)
    tau x  = maybe 0 (/ mx) (lookup x plT)
    tbar x = maybe 0 id (lookup x belT)

expertA, expertB, expertC :: SubjModel Engine
expertA = mkExpert [(EOK,0.3),(EFilter,0.9),(EBearing,0.5),(ECrit,0.1)]
                   [(EOK,0.1),(EFilter,0.0),(EBearing,0.2),(ECrit,0.6)]
expertB = mkExpert [(EOK,0.2),(EFilter,0.7),(EBearing,0.8),(ECrit,0.3)]
                   [(EOK,0.2),(EFilter,0.1),(EBearing,0.0),(ECrit,0.5)]
expertC = mkExpert [(EOK,0.1),(EFilter,0.6),(EBearing,0.9),(ECrit,0.4)]
                   [(EOK,0.3),(EFilter,0.2),(EBearing,0.0),(ECrit,0.4)]

-- Комбинирование экспертов: взвешенная сумма tau/tauBar, затем нормировка sup tau = 1
combineExperts :: [(SubjModel Engine, Double)] -> SubjModel Engine
combineExperts ws = SubjModel engStates tau tbar
  where
    wsum      = sum (map snd ws)
    rawTau x  = sum [w * smTau m x    | (m, w) <- ws] / wsum
    rawTBar x = sum [w * smTauBar m x | (m, w) <- ws] / wsum
    mx        = maximum (map rawTau engStates)
    tau x     = if mx == 0 then rawTau x else rawTau x / mx
    tbar      = rawTBar

combined :: SubjModel Engine
combined = combineExperts [(expertA, 0.4), (expertB, 0.35), (expertC, 0.25)]

-- Условная модель: ограничить домен наблюдаемым событием и перенормировать
conditionModel :: SubjModel Engine -> [Engine] -> SubjModel Engine
conditionModel m obs = SubjModel obs tau (smTauBar m)
  where
    mx    = maximum (0 : map (smTau m) obs)
    tau x = if mx == 0 then smTau m x else smTau m x / mx

-- Серьёзность состояния (функция потерь для интеграла Сугено)
severity :: Engine -> Double
severity EOK      = 0.0
severity EFilter  = 0.3
severity EBearing = 0.6
severity ECrit    = 1.0

demo_engine :: IO ()
demo_engine = do
  putStrLn "=== Диагностика двигателя: 3 эксперта ==="
  let warn     = [EFilter, EBearing]
      compWarn = filter (`notElem` warn) engStates
  putStrLn $ "Pl({Filter,Bearing})  = " ++ show (plMeasure warn (smTau combined))
  putStrLn $ "Bel({Filter,Bearing}) = " ++ show (belMeasure compWarn (smTauBar combined))
  putStrLn $ "Зазор Pl - Bel        = "
           ++ show (plMeasure warn (smTau combined) - belMeasure compWarn (smTauBar combined))
  putStrLn ""
  putStrLn "-- Интегралы риска (severity) --"
  putStrLn $ "Pl-интеграл (оптимист):   " ++ show (plIntegral engStates (smTau combined) severity)
  putStrLn $ "Bel-интеграл (пессимист): " ++ show (belIntegral engStates (smTauBar combined) severity)
  putStrLn ""
  putStrLn "-- Принцип относительности (gamma = sqrt сохраняет порядок) --"
  let g = combined { smTau = sqrt . smTau combined }  -- принцип относительности
  putStrLn $ "Pl({Filter}) до:    " ++ show (plMeasure [EFilter] (smTau combined))
  putStrLn $ "Pl({Filter}) после: " ++ show (plMeasure [EFilter] (smTau g))
  putStrLn ""
  putStrLn "-- Условная модель (датчик исключил Critical) --"
  let c = conditionModel combined [EOK, EFilter, EBearing]
  putStrLn $ "Pl(Bearing | not Crit) = " ++ show (plMeasure [EBearing] (smTau c))
  putStrLn $ "Pl(Filter  | not Crit) = " ++ show (plMeasure [EFilter]  (smTau c))

demo_engine

Line 15: Use fromMaybe
Found:
maybe 0 id
Why not:
Data.Maybe.fromMaybe 0Line 53: Use camelCase
Found:
demo_engine :: IO ()
Why not:
demoEngine :: IO ()Line 54: Use camelCase
Found:
demo_engine = ...
Why not:
demoEngine = ...

=== Диагностика двигателя: 3 эксперта ===
Pl({Filter,Bearing})  = 1.0
Bel({Filter,Bearing}) = 0.185
Зазор Pl - Bel        = 0.815

-- Интегралы риска (severity) --
Pl-интеграл (оптимист):   0.6
Bel-интеграл (пессимист): 0.185

-- Принцип относительности (gamma = sqrt сохраняет порядок) --
Pl({Filter}) до:    1.0
Pl({Filter}) после: 1.0

-- Условная модель (датчик исключил Critical) --
Pl(Bearing | not Crit) = 0.9419252187748607
Pl(Filter  | not Crit) = 1.0

# ✅ Итоги: Теория Субъективного Моделирования Пытьева

## Основные результаты

| Раздел | Ключевое понятие | Источник |
|--------|-----------------|----------|
| 1 | Пространство $(X, \mathcal{P}(X), \mathrm{Pl}^{\tilde{x}}, \mathrm{Bel}^{\tilde{x}})$ — субъективная модель НОЭ | Часть 1, п. 1.1 |
| 2 | Шкалы $L = ([0,1], \max, \min)$ и $\bar{L} = ([0,1], \min, \max)$ | Часть 1, п. 1.3 |
| 3 | Меры Pl и Bel через $\sup$ и $\inf$ распределений $\tau, \bar{\tau}$ | Часть 1, п. 1.1 |
| 4 | Принцип относительности — группа $\Gamma$ автоморфизмов | Часть 1, п. 1.3 |
| 5 | Pl- и bel-интегралы: $\sup\min$ и $\inf\max$ (Теорема 1.1) | Часть 1, п. 1.6 |
| 6 | Субъективная независимость $\equiv$ Pl-независимость | Часть 1, п. 1.7 |
| 7 | Условные распределения через субъективные шкалы | Часть 1, п. 1.8 |
| 8 | Три варианта мер: основной, с фикс. точками, психофизический | Часть 1, п. 1.9 |
| 9 | Эмпирическое восстановление НЧ.НОЭ по наблюдениям | Часть 1, разд. 2 |
| 10 | Комбинирование через матрицы парных сравнений | Часть 1, п. 2.2 |
| 11 | Энтропии: $H(\tau) = \mathrm{pl}(\bar{\tau})$, $H(\bar{\tau}) = \mathrm{bel}(\tau)$ | Часть 2, разд. 2 |
| 12 | Оптимальное правило идентификации НО.НЧ.О. | Часть 2, разд. 3 |
| 13 | $[0,1]$ — quantale/фрейм; топологии Скотта $\mathcal{T}_{\uparrow}, \mathcal{T}_{\downarrow}$ индуцируют битопос | Категорное |
| 14 | $\mathrm{Lan}_J\,\tau = \mathrm{Pl} = \mathrm{Int}_{\mathcal{T}_{\uparrow}}$; гипотеза: единство через Isbell duality над $[0,1]$ | Гипотеза |

## Ключевые формулы

$$\mathrm{Pl}(E) = \sup_{x \in E} \tau(x), \qquad \mathrm{Bel}(E) = \inf_{x \notin E} \bar{\tau}(x)$$

$$\mathrm{pl}_{\tau}(g) = \sup_x \min(\tau(x), g(x)), \qquad \mathrm{bel}_{\bar{\tau}}(\bar{g}) = \inf_x \max(\bar{\tau}(x), \bar{g}(x))$$

$$H(\tau) = \mathrm{pl}_{\tau}(\bar{\tau}), \qquad H(\bar{\tau}) = \mathrm{bel}_{\bar{\tau}}(\tau)$$

$$\mathrm{Lan}_J\,\tau\,(E) = \int^{x} J(x,E) \otimes_{\min} \tau(x) = \sup_{x \in E}\tau(x) = \mathrm{Pl}(E)$$

## Три вида знания

| Модель | $\tau(x)$ для всех $x$ | $\bar{\tau}(x)$ для всех $x$ |
|--------|----------------------|-----------------------------|
| Абсолютное незнание | $1$ | $0$ |
| Точное знание $x_0$ | $\mathbf{1}_{x=x_0}$ | $\mathbf{1}_{x=x_0}$ |
| Произвольная модель | $\tau: X \to [0,1]$ | $\bar{\tau}: X \to [0,1]$ |

## Статус категорной гипотезы

| Утверждение | Статус |
|-------------|--------|
| Coend = sup = Interior на дискретном $X$ | ✅ Доказано |
| End = inf = Closure на дискретном $X$ | ✅ Доказано |
| $[0,1]$ — quantale (фрейм) | ✅ Доказано |
| Isbell adjunction $\mathcal{O} \dashv \mathrm{Spec}$ над $[0,1]$ = пара (Pl, Bel) | ⚓ Гипотеза |

В разделах 15–16 теория проверена делом: три сравнительных примера (битопос vs расширения Кана) и практическая диагностика двигателя тремя экспертами.


---

**← Предыдущий:** [Неопределённость](Uncertainty.ipynb)
